In [3]:
import serial
import time
import struct

class ModbusRaw:
    """Класс для работы с Modbus RTU через сырые байты"""
    
    def __init__(self, port, baudrate=9600, timeout=1):
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.ser = None
    
    def open(self):
        """Открывает порт"""
        self.ser = serial.Serial(
            port=self.port,
            baudrate=self.baudrate,
            timeout=self.timeout,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            bytesize=serial.EIGHTBITS
        )
        print(f"✅ Порт {self.port} открыт")
        return self.ser
    
    def close(self):
        """Закрывает порт"""
        if self.ser and self.ser.is_open:
            self.ser.close()
            print("✅ Порт закрыт")
    
    def send_raw(self, data):
        """Отправляет сырые байты"""
        if not self.ser or not self.ser.is_open:
            raise Exception("Порт не открыт")
        
        # Отправляем байты
        bytes_sent = self.ser.write(data)
        print(f"📤 Отправлено {bytes_sent} байт: {data.hex().upper()}")
        return bytes_sent
    
    def read_raw(self, size=256):
        """Читает сырые байты"""
        if not self.ser or not self.ser.is_open:
            raise Exception("Порт не открыт")
        
        # Ждем ответ
        time.sleep(0.1)
        
        # Читаем все доступные байты
        response = self.ser.read(size)
        if response:
            print(f"📥 Получено {len(response)} байт: {response.hex().upper()}")
        else:
            print("⏰ Нет ответа (таймаут)")
        
        return response
    
    def send_and_receive(self, data, read_size=256):
        """Отправляет и сразу читает ответ"""
        self.send_raw(data)
        return self.read_raw(read_size)

# ============================================
# ФУНКЦИИ ДЛЯ РАБОТЫ С MODBUS RTU
# ============================================

def calculate_crc(data):
    """Вычисляет CRC16 для Modbus RTU"""
    crc = 0xFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            if crc & 0x0001:
                crc >>= 1
                crc ^= 0xA001
            else:
                crc >>= 1
    return crc

def build_modbus_request(slave_id, function, address, count=1):
    """Строит Modbus RTU запрос с CRC"""
    # Формируем тело запроса
    request = bytes([
        slave_id,           # Адрес ведомого
        function,           # Функция (03 - чтение регистров)
        (address >> 8) & 0xFF,  # Старший байт адреса
        address & 0xFF,     # Младший байт адреса
        (count >> 8) & 0xFF,    # Старший байт количества
        count & 0xFF        # Младший байт количества
    ])
    
    # Вычисляем CRC
    crc = calculate_crc(request)
    
    # Добавляем CRC в конец (младший байт сначала)
    request_with_crc = request + bytes([crc & 0xFF, (crc >> 8) & 0xFF])
    
    return request_with_crc

def parse_modbus_response(response):
    """Парсит Modbus RTU ответ"""
    if len(response) < 3:
        return None
    
    slave_id = response[0]
    function = response[1]

    print(response)
    # Проверяем на ошибку
    if function & 0x80:
        error_code = response[2]
        return {
            'type': 'error',
            'slave_id': slave_id,
            'function': function,
            'error_code': error_code
        }
    
    # Нормальный ответ
    data_length = response[2]
    data = response[3:3+data_length]
    crc_received = response[-2:] if len(response) >= 5 else None
    
    # Проверяем CRC
    if crc_received:
        crc_calc = calculate_crc(response[:-2])
        crc_calc_bytes = bytes([crc_calc & 0xFF, (crc_calc >> 8) & 0xFF])
        crc_ok = crc_received == crc_calc_bytes
        
        if not crc_ok:
            print(f"⚠️ CRC не совпадает! Получено: {crc_received.hex().upper()}, "
                  f"рассчитано: {crc_calc_bytes.hex().upper()}")
    
    return {
        'type': 'response',
        'slave_id': slave_id,
        'function': function,
        'data_length': data_length,
        'data': data,
        'data_hex': data.hex().upper(),
        'crc': crc_received,
        'crc_ok': crc_ok if crc_received else None
    }

# ============================================
# ПРИМЕРЫ ИСПОЛЬЗОВАНИЯ
# ============================================

def example_1_simple_request():
    """Пример 1: Простой запрос сырыми байтами"""
    
    print("\n" + "="*50)
    print("ПРИМЕР 1: Отправка сырых байт")
    print("="*50)
    
    # Создаем экземпляр
    modbus = ModbusRaw('COM5', 9600)
    
    try:
        # Открываем порт
        modbus.open()
        
        # Отправляем сырые байты (запрос для адреса 18, регистр 0)
        raw_data = bytes.fromhex('12 03 00 00 00 01 4F E5')
        response = modbus.send_and_receive(raw_data)
        
        if response:
            # Парсим ответ
            parsed = parse_modbus_response(response)
            if parsed:
                print(f"\n📊 Парсинг ответа:")
                print(f"  Адрес: {parsed['slave_id']}")
                print(f"  Функция: {parsed['function']}")
                if parsed['type'] == 'response':
                    print(f"  Данные: {parsed['data_hex']}")
                    if parsed['data']:
                        value = int.from_bytes(parsed['data'], byteorder='big')
                        print(f"  Значение: {value}")
                elif parsed['type'] == 'error':
                    print(f"  Ошибка: {parsed['error_code']}")
    
    finally:
        modbus.close()

def example_2_build_request():
    """Пример 2: Построение запроса с автоматическим CRC"""
    
    print("\n" + "="*50)
    print("ПРИМЕР 2: Построение запроса с CRC")
    print("="*50)
    
    modbus = ModbusRaw('COM5', 9600)
    
    try:
        modbus.open()
        
        # Строим запрос для чтения регистра 0 (адрес 18)
        request = build_modbus_request(
            slave_id=18,
            function=3,      # 03 - чтение holding registers
            address=201,       # регистр 1
            count=1          # читаем 1 регистр
        )
        
        print(f"📤 Построенный запрос: {request.hex().upper()}")
        
        # Отправляем
        response = modbus.send_and_receive(request)
        
        if response:
            parsed = parse_modbus_response(response)
            if parsed and parsed['type'] == 'response':
                value = int.from_bytes(parsed['data'], byteorder='big')
                print(f"\n✅ Значение регистра 1: {value}")
    
    finally:
        modbus.close()



# ============================================
# ЗАПУСК ПРИМЕРОВ
# ============================================

if __name__ == "__main__":
    # Выберите пример для запуска
    # example_1_simple_request()
    example_2_build_request()
    # example_3_multiple_requests()
    # example_4_custom_bytes()
    # example_5_write_register()
    
    # Запускаем основной пример
    #example_1_simple_request()


ПРИМЕР 2: Построение запроса с CRC
✅ Порт COM5 открыт
📤 Построенный запрос: 120300C900015697
📤 Отправлено 8 байт: 120300C900015697
📥 Получено 9 байт: 120302384500004C47
b'\x12\x03\x028E\x00\x00LG'

✅ Значение регистра 1: 14405
✅ Порт закрыт


In [12]:
import minimalmodbus

# 1. Укажите порт (например, /dev/ttyUSB0) и адрес устройства (обычно 1)
instrument = minimalmodbus.Instrument('COM5', 18)

# 2. Настройте параметры связи (скорость, таймаут) в соответствии с вашим устройством
instrument.serial.baudrate = 9600
instrument.serial.timeout = 0.5

# 3. Укажите, что используете режим RTU (он используется по умолчанию)
instrument.mode = minimalmodbus.MODE_RTU

try:
    # 4. Прочитайте регистр (номер 289, с одним десятичным знаком)
    # Если значение должно быть с плавающей точкой
    temperature = instrument.read_float(313, 3, 2)  # 3 = 32-bit float, 2 = big endianinstrument.read_register(201, 1)
    print(f"Температура: {temperature} °C")
except Exception as e:
    print(f"Ошибка при чтении: {e}")

Температура: 0.05890011042356491 °C


In [8]:
import serial
import time

ser = serial.Serial('COM5', 9600, timeout=1)

# Запрос: адрес 18, функция 03, регистр 0, кол-во 1
# [0x12, 0x03, 0x00, 0x00, 0x00, 0x01, 0x4F, 0xE5]  # CRC для адреса 18
request = bytes.fromhex('12 03 01 39 00 01 4F E5')
ser.write(request)
time.sleep(0.1)

response = ser.read(20)
print(f"Ответ: {response.hex()}")

# Ваш ответ: 12 03 02 00 E5 00 00
# Данные: 00 E5 = 229

SerialException: could not open port 'COM5': PermissionError(13, 'Отказано в доступе.', None, 5)

In [4]:
from Serial import serial
import time

try:
    ser = serial.Serial('COM5', 9600, timeout=2)
    print("Порт открыт!")
    
    # Отправляем тестовый запрос
    request = bytes.fromhex('12 03 00 00 00 01 4F E5')
    ser.write(request)
    time.sleep(0.1)
    
    response = ser.read(20)
    print(f"Ответ: {response.hex()}")
    
    ser.close()  # Важно: закрываем порт!
    
except PermissionError:
    print("❌ Ошибка доступа! Проверьте:")
    print("1. Закрыты ли другие программы (Modbus Poll, Putty и т.д.)")
    print("2. Не запущен ли другой Python-скрипт")
    print("3. Попробуйте перезагрузить компьютер")
except Exception as e:
    print(f"❌ Другая ошибка: {e}")

ModuleNotFoundError: No module named 'Serial'

In [1]:
import time

# Пытаемся импортировать правильно
try:
    import serial
    # Проверяем, что класс Serial существует
    if hasattr(serial, 'Serial'):
        print(f"✅ pyserial версия {serial.__version__} успешно загружена")
    else:
        raise ImportError("Нет атрибута Serial")
except ImportError as e:
    print("❌ Ошибка импорта serial. Попробуйте:")
    print("1. pip uninstall serial")
    print("2. pip install pyserial")
    raise e

# Теперь работаем с портом
def find_and_open_port(port_name='COM5'):
    """Находит и открывает порт"""
    
    # Проверяем доступные порты
    import serial.tools.list_ports
    ports = serial.tools.list_ports.comports()
    print("Доступные порты:")
    for port in ports:
        print(f"  {port.device} - {port.description}")
    
    # Открываем порт
    for attempt in range(5):
        try:
            ser = serial.Serial(
                port=port_name,
                baudrate=9600,
                timeout=1,
                parity=serial.PARITY_NONE,
                stopbits=serial.STOPBITS_ONE,
                bytesize=serial.EIGHTBITS
            )
            print(f"✅ Порт {port_name} успешно открыт!")
            return ser
        except serial.SerialException as e:
            print(f"❌ Попытка {attempt+1}/5: {e}")
            time.sleep(2)
    
    raise Exception(f"Не удалось открыть порт {port_name}")

# Использование
if __name__ == "__main__":
    try:
        # Открываем порт
        ser = find_and_open_port('COM5')
        
        # Отправляем Modbus запрос (адрес 18, функция 03, регистр 0)
        request = bytes.fromhex('12 03 00 00 00 01 4F E5')
        ser.write(request)
        time.sleep(0.1)
        
        # Читаем ответ
        response = ser.read(20)
        print(f"Ответ: {response.hex()}")
        
        # Закрываем порт
        ser.close()
        print("✅ Порт закрыт")
        
    except Exception as e:
        print(f"❌ Ошибка: {e}")

❌ Ошибка импорта serial. Попробуйте:
1. pip uninstall serial
2. pip install pyserial


ImportError: Нет атрибута Serial

In [1]:
import serial

print(f"Версия: {serial.__version__}")
print(f"Доступные атрибуты: {[x for x in dir(serial) if not x.startswith('_')]}")

Версия: 3.5
Доступные атрибуты: ['CR', 'EIGHTBITS', 'FIVEBITS', 'LF', 'PARITY_EVEN', 'PARITY_MARK', 'PARITY_NAMES', 'PARITY_NONE', 'PARITY_ODD', 'PARITY_SPACE', 'PortNotOpenError', 'SEVENBITS', 'SIXBITS', 'STOPBITS_ONE', 'STOPBITS_ONE_POINT_FIVE', 'STOPBITS_TWO', 'Serial', 'SerialBase', 'SerialException', 'SerialTimeoutException', 'Timeout', 'VERSION', 'XOFF', 'XON', 'absolute_import', 'basestring', 'importlib', 'io', 'iterbytes', 'os', 'protocol_handler_packages', 'serial_for_url', 'serialutil', 'serialwin32', 'sys', 'time', 'to_bytes', 'unicode', 'win32']


In [2]:
import time
import sys

# Проверка установки
try:
    import serial
    if not hasattr(serial, 'Serial'):
        print("❌ ОШИБКА: Установлен пакет 'serial', а нужно 'pyserial'")
        print("Выполните команды:")
        print("pip uninstall serial pyserial -y")
        print("pip install pyserial")
        sys.exit(1)
    
    print(f"✅ pyserial версия {serial.__version__} успешно загружена")
    
except ImportError as e:
    print(f"❌ Ошибка импорта: {e}")
    print("Выполните: pip install pyserial")
    sys.exit(1)

# Теперь работаем с портом
def find_and_open_port(port_name='COM5'):
    """Находит и открывает порт"""
    
    # Проверяем доступные порты
    try:
        import serial.tools.list_ports
        ports = serial.tools.list_ports.comports()
        print("\nДоступные порты:")
        for port in ports:
            print(f"  {port.device} - {port.description}")
    except:
        pass
    
    # Открываем порт с повторными попытками
    for attempt in range(5):
        try:
            ser = serial.Serial(
                port=port_name,
                baudrate=9600,
                timeout=2,
                parity=serial.PARITY_NONE,
                stopbits=serial.STOPBITS_ONE,
                bytesize=serial.EIGHTBITS
            )
            print(f"\n✅ Порт {port_name} успешно открыт!")
            return ser
            
        except serial.SerialException as e:
            print(f"❌ Попытка {attempt+1}/5: {e}")
            time.sleep(2)
    
    raise Exception(f"Не удалось открыть порт {port_name}")

# Основная функция
def main():
    ser = None
    try:
        # Открываем порт
        ser = find_and_open_port('COM5')
        
        # Отправляем Modbus запрос (адрес 18, функция 03, регистр 0)
        print("\nОтправка запроса...")
        request = bytes.fromhex('12 03 00 01 00 01 4F E5')
        ser.write(request)
        time.sleep(0.2)
        
        # Читаем ответ
        response = ser.read(20)
        if response:
            print(f"✅ Получен ответ: {response.hex()}")
            print(f"Длина: {len(response)} байт")
            
            # Парсим ответ вручную
            if len(response) >= 5:
                slave_id = response[0]
                function = response[1]
                data_len = response[2]
                data = response[3:3+data_len]
                
                print(f"\nПарсинг:")
                print(f"  Адрес: {slave_id} (0x{slave_id:02X})")
                print(f"  Функция: {function}")
                print(f"  Длина данных: {data_len}")
                print(f"  Данные: {data.hex()}")
                if data:
                    value = int.from_bytes(data, byteorder='big')
                    print(f"  Значение: {value}")
        else:
            print("❌ Нет ответа от устройства")
            
    except Exception as e:
        print(f"\n❌ Ошибка: {e}")
        
    finally:
        if ser and ser.is_open:
            ser.close()
            print("\n✅ Порт закрыт")

if __name__ == "__main__":
    main()

✅ pyserial версия 3.5 успешно загружена

Доступные порты:
  COM3 - Стандартный последовательный порт по соединению Bluetooth (COM3)
  COM4 - Стандартный последовательный порт по соединению Bluetooth (COM4)
  COM5 - USB-SERIAL CH340 (COM5)

✅ Порт COM5 успешно открыт!

Отправка запроса...
❌ Нет ответа от устройства

✅ Порт закрыт


In [2]:
import serial
import time
import serial.tools.list_ports

def scan_modbus_devices():
    """Сканирует все возможные настройки"""
    
    # Параметры для перебора
    baudrates = [9600, 19200, 38400, 57600, 115200, 4800, 2400]
    parities = [serial.PARITY_NONE, serial.PARITY_EVEN, serial.PARITY_ODD]
    stopbits = [serial.STOPBITS_ONE, serial.STOPBITS_TWO]
    
    # Запросы для разных адресов (широковещательный и адрес 1)
    requests = [
        #bytes.fromhex('01 03 00 00 00 01 84 0A'),  # Адрес 1, регистр 0
        #bytes.fromhex('0A 03 00 00 00 01 85 0A'),  # Адрес 10
        bytes.fromhex('12 03 00 01 00 01 4F E5'),  # Адрес 18 (как в вашем ответе)
        #bytes.fromhex('FF 03 00 00 00 01 91 D4'),  # Широковещательный адрес
    ]
    
    print("🔍 Сканирование Modbus устройств...")
    print("Это может занять некоторое время\n")
    
    for baud in baudrates:
        for parity in parities:
            for stop in stopbits:
                for req in requests:
                    try:
                        ser = serial.Serial(
                            port='COM5',
                            baudrate=baud,
                            timeout=1,
                            parity=parity,
                            stopbits=stop,
                            bytesize=serial.EIGHTBITS
                        )
                        
                        # Отправляем запрос
                        ser.write(req)
                        time.sleep(0.2)
                        
                        # Читаем ответ
                        response = ser.read(20)
                        ser.close()
                        
                        if response and len(response) >= 3:
                            print(f"✅ НАЙДЕНО!")
                            print(f"  Скорость: {baud}")
                            print(f"  Четность: {parity}")
                            print(f"  Стоп-биты: {stop}")
                            print(f"  Запрос: {req.hex()}")
                            print(f"  Ответ: {response.hex()}")
                            print(f"  Длина: {len(response)} байт\n")
                            return True
                            
                    except Exception as e:
                        continue
    
    print("❌ Устройство не найдено ни с одним набором параметров")
    return False

if __name__ == "__main__":
    scan_modbus_devices()

🔍 Сканирование Modbus устройств...
Это может занять некоторое время

❌ Устройство не найдено ни с одним набором параметров


In [2]:
import serial
import time
import minimalmodbus

def test_with_minimalmodbus():
    """Тестируем через minimalmodbus"""
    
    try:
        # Пробуем разные адреса
        for slave_id in [1, 10, 18, 255]:
            print(f"\n🔍 Пробуем адрес {slave_id}...")
            
            try:
                instrument = minimalmodbus.Instrument('COM5', slave_id)
                instrument.serial.baudrate = 9600
                instrument.serial.timeout = 2
                instrument.mode = minimalmodbus.MODE_RTU
                
                # Пробуем прочитать регистр 0
                value = instrument.read_register(1, 0, 3, False)
                print(f"  ✅ Успех! Адрес {slave_id}, значение: {value}")
                return True
                
            except minimalmodbus.NoResponseError:
                print(f"  ⚠️ Нет ответа от адреса {slave_id}")
            except Exception as e:
                print(f"  ❌ Ошибка: {e}")
        
        print("\n❌ Ни один адрес не ответил")
        return False
        
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return False

if __name__ == "__main__":
    test_with_minimalmodbus()


🔍 Пробуем адрес 1...
  ⚠️ Нет ответа от адреса 1

🔍 Пробуем адрес 10...
  ⚠️ Нет ответа от адреса 10

🔍 Пробуем адрес 18...
  ✅ Успех! Адрес 18, значение: 4


In [7]:
import tkinter as tk
from tkinter import ttk, messagebox
import threading
import time
import serial
import minimalmodbus

class ModbusMonitorApp:
    """Приложение для мониторинга Modbus регистров"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Modbus RTU Монитор регистров")
        self.root.geometry("900x700")
        
        # Переменные для подключения
        self.port = tk.StringVar(value="COM5")
        self.slave_id = tk.IntVar(value=18)
        self.baudrate = tk.IntVar(value=9600)
        self.timeout = tk.DoubleVar(value=1.0)
        
        # Переменные для мониторинга
        self.start_reg = tk.IntVar(value=0)
        self.reg_count = tk.IntVar(value=10)
        self.update_interval = tk.IntVar(value=1000)  # мс
        self.is_monitoring = False
        self.monitor_thread = None
        self.instrument = None
        
        # Список регистров для отображения
        self.register_data = {}
        
        # Создаем интерфейс
        self.create_widgets()
        
    def create_widgets(self):
        """Создает все виджеты интерфейса"""
        
        # ===== Верхняя панель настроек =====
        settings_frame = ttk.LabelFrame(self.root, text="Настройки подключения", padding=10)
        settings_frame.pack(fill=tk.X, padx=10, pady=5)
        
        # Порт
        ttk.Label(settings_frame, text="Порт:").grid(row=0, column=0, padx=5, sticky=tk.W)
        ttk.Entry(settings_frame, textvariable=self.port, width=10).grid(row=0, column=1, padx=5)
        
        # Slave ID
        ttk.Label(settings_frame, text="Адрес:").grid(row=0, column=2, padx=5, sticky=tk.W)
        ttk.Entry(settings_frame, textvariable=self.slave_id, width=10).grid(row=0, column=3, padx=5)
        
        # Скорость
        ttk.Label(settings_frame, text="Скорость:").grid(row=0, column=4, padx=5, sticky=tk.W)
        baud_combo = ttk.Combobox(settings_frame, textvariable=self.baudrate, 
                                  values=[2400, 4800, 9600, 19200, 38400, 57600, 115200],
                                  width=10)
        baud_combo.grid(row=0, column=5, padx=5)
        
        # Таймаут
        ttk.Label(settings_frame, text="Таймаут:").grid(row=0, column=6, padx=5, sticky=tk.W)
        ttk.Entry(settings_frame, textvariable=self.timeout, width=10).grid(row=0, column=7, padx=5)
        
        # Кнопка подключения
        self.connect_btn = ttk.Button(settings_frame, text="Подключиться", 
                                      command=self.toggle_connection)
        self.connect_btn.grid(row=0, column=8, padx=10)
        
        # ===== Панель настроек мониторинга =====
        monitor_frame = ttk.LabelFrame(self.root, text="Настройки мониторинга", padding=10)
        monitor_frame.pack(fill=tk.X, padx=10, pady=5)
        
        # Начальный регистр
        ttk.Label(monitor_frame, text="Начальный регистр:").grid(row=0, column=0, padx=5, sticky=tk.W)
        ttk.Entry(monitor_frame, textvariable=self.start_reg, width=10).grid(row=0, column=1, padx=5)
        
        # Количество регистров
        ttk.Label(monitor_frame, text="Количество:").grid(row=0, column=2, padx=5, sticky=tk.W)
        ttk.Entry(monitor_frame, textvariable=self.reg_count, width=10).grid(row=0, column=3, padx=5)
        
        # Интервал обновления
        ttk.Label(monitor_frame, text="Интервал (мс):").grid(row=0, column=4, padx=5, sticky=tk.W)
        ttk.Entry(monitor_frame, textvariable=self.update_interval, width=10).grid(row=0, column=5, padx=5)
        
        # Кнопки управления
        self.start_btn = ttk.Button(monitor_frame, text="▶ Старт", 
                                    command=self.start_monitoring)
        self.start_btn.grid(row=0, column=6, padx=10)
        
        self.stop_btn = ttk.Button(monitor_frame, text="⏹ Стоп", 
                                   command=self.stop_monitoring, state=tk.DISABLED)
        self.stop_btn.grid(row=0, column=7, padx=10)
        
        ttk.Button(monitor_frame, text="🔄 Обновить", 
                  command=self.update_table).grid(row=0, column=8, padx=10)
        
        # ===== Таблица с данными =====
        table_frame = ttk.LabelFrame(self.root, text="Данные регистров", padding=10)
        table_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        
        # Создаем Treeview с прокруткой
        self.tree = ttk.Treeview(table_frame, columns=('value', 'hex', 'bin', 'float'), 
                                 show='headings', height=20)
        
        # Настройка колонок
        self.tree.heading('value', text='Десятичное')
        self.tree.heading('hex', text='Hex')
        self.tree.heading('bin', text='Бинарное')
        self.tree.heading('float', text='Float (если применимо)')
        
        self.tree.column('value', width=120, anchor=tk.CENTER)
        self.tree.column('hex', width=100, anchor=tk.CENTER)
        self.tree.column('bin', width=150, anchor=tk.CENTER)
        self.tree.column('float', width=150, anchor=tk.CENTER)
        
        # Добавляем скроллбар
        scrollbar = ttk.Scrollbar(table_frame, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        
        # Размещение
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # ===== Статусная строка =====
        self.status_var = tk.StringVar(value="Готов к работе")
        status_bar = ttk.Label(self.root, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W)
        status_bar.pack(fill=tk.X, padx=10, pady=5)
        
        # ===== Информационная панель =====
        info_frame = ttk.LabelFrame(self.root, text="Информация", padding=10)
        info_frame.pack(fill=tk.X, padx=10, pady=5)
        
        self.info_text = tk.Text(info_frame, height=4, width=80, wrap=tk.WORD)
        self.info_text.pack(fill=tk.X)
        
    def toggle_connection(self):
        """Подключается или отключается от устройства"""
        if self.instrument is None:
            self.connect_device()
        else:
            self.disconnect_device()
    
    def connect_device(self):
        """Подключается к Modbus устройству"""
        try:
            self.instrument = minimalmodbus.Instrument(
                self.port.get(),
                self.slave_id.get()
            )
            self.instrument.serial.baudrate = self.baudrate.get()
            self.instrument.serial.timeout = self.timeout.get()
            self.instrument.mode = minimalmodbus.MODE_RTU
            
            # Проверяем подключение
            self.instrument.read_register(1, 0, 3, False)
            
            self.connect_btn.config(text="Отключиться")
            self.status_var.set(f"Подключено к {self.port.get()}, адрес {self.slave_id.get()}")
            self.log_info(f"✅ Подключено к устройству на порту {self.port.get()}")
            self.start_btn.config(state=tk.NORMAL)
            
        except Exception as e:
            self.status_var.set(f"❌ Ошибка подключения: {e}")
            print(f"❌ Ошибка подключения: {e}")
            self.log_info(f"❌ Ошибка: {e}")
            self.instrument = None
            messagebox.showerror("Ошибка", f"Не удалось подключиться:\n{e}")
    
    def disconnect_device(self):
        """Отключается от устройства"""
        if self.is_monitoring:
            self.stop_monitoring()
        
        if self.instrument:
            try:
                self.instrument.serial.close()
            except:
                pass
            self.instrument = None
        
        self.connect_btn.config(text="Подключиться")
        self.start_btn.config(state=tk.DISABLED)
        self.status_var.set("Отключено")
        self.log_info("🔌 Отключено от устройства")
    
    def start_monitoring(self):
        """Запускает мониторинг в отдельном потоке"""
        if self.instrument is None:
            messagebox.showwarning("Предупреждение", "Сначала подключитесь к устройству")
            return
        
        if self.is_monitoring:
            return
        
        try:
            # Проверяем параметры
            start = self.start_reg.get()
            count = self.reg_count.get()
            
            if count < 1 or count > 100:
                messagebox.showwarning("Предупреждение", "Количество регистров должно быть от 1 до 100")
                return
            
            if start < 0 or start > 10000:
                messagebox.showwarning("Предупреждение", "Начальный регистр должен быть от 0 до 10000")
                return
            
            # Очищаем таблицу
            for item in self.tree.get_children():
                self.tree.delete(item)
            
            # Добавляем строки для регистров
            for i in range(count):
                self.tree.insert('', tk.END, iid=f"reg_{start+i}", 
                               values=(f"???", f"0x0000", f"00000000", "—"))
            
            self.is_monitoring = True
            self.start_btn.config(state=tk.DISABLED)
            self.stop_btn.config(state=tk.NORMAL)
            self.status_var.set(f"🔄 Мониторинг регистров {start}-{start+count-1}")
            self.log_info(f"▶ Запущен мониторинг: регистры {start}-{start+count-1}, "
                         f"интервал {self.update_interval.get()} мс")
            
            # Запускаем поток мониторинга
            self.monitor_thread = threading.Thread(target=self.monitor_loop, daemon=True)
            self.monitor_thread.start()
            
        except Exception as e:
            self.status_var.set(f"❌ Ошибка: {e}")
            self.log_info(f"❌ Ошибка: {e}")
    
    def stop_monitoring(self):
        """Останавливает мониторинг"""
        self.is_monitoring = False
        if self.monitor_thread:
            self.monitor_thread.join(timeout=1)
            self.monitor_thread = None
        
        self.start_btn.config(state=tk.NORMAL)
        self.stop_btn.config(state=tk.DISABLED)
        self.status_var.set("Мониторинг остановлен")
        self.log_info("⏹ Мониторинг остановлен")
    
    def monitor_loop(self):
        """Цикл мониторинга (работает в отдельном потоке)"""
        start = self.start_reg.get()
        count = self.reg_count.get()
        interval = self.update_interval.get() / 1000.0
        
        while self.is_monitoring:
            try:
                # Читаем все регистры за один раз
                if self.instrument:
                    # Используем read_registers для эффективности
                    try:
                        values = self.instrument.read_registers(start, count, 3, False)
                        
                        # Обновляем таблицу в главном потоке
                        self.root.after(0, self.update_register_values, start, values)
                        
                        # Обновляем статус
                        self.root.after(0, lambda: self.status_var.set(
                            f"🔄 Последнее обновление: {time.strftime('%H:%M:%S')}"
                        ))
                        
                    except Exception as e:
                        self.root.after(0, lambda: self.status_var.set(f"⚠️ Ошибка чтения: {e}"))
                        self.root.after(0, lambda: self.log_info(f"⚠️ Ошибка: {e}"))
                
                # Ждем до следующего обновления
                time.sleep(interval)
                
            except Exception as e:
                self.root.after(0, lambda: self.log_info(f"❌ Ошибка в цикле: {e}"))
                time.sleep(1)
    
    def update_register_values(self, start_reg, values):
        """Обновляет значения в таблице"""
        for i, value in enumerate(values):
            reg_num = start_reg + i
            item_id = f"reg_{reg_num}"
            
            if self.tree.exists(item_id):
                # Форматируем значения
                hex_val = f"0x{value:04X}"
                bin_val = format(value, '016b')
                
                # Пытаемся интерпретировать как float (если возможно)
                float_val = "—"
                try:
                    # Если значение может быть float (например, 1234 -> 12.34)
                    if value > 0:
                        # Простая эвристика: если значение > 1000, возможно это float * 100
                        if value > 10000:
                            float_val = f"{value / 10000:.4f}"
                        elif value > 1000:
                            float_val = f"{value / 100:.2f}"
                        elif value > 100:
                            float_val = f"{value / 10:.1f}"
                except:
                    pass
                
                self.tree.item(item_id, values=(value, hex_val, bin_val, float_val))
    
    def update_table(self):
        """Принудительное обновление таблицы"""
        if self.instrument is None:
            messagebox.showwarning("Предупреждение", "Сначала подключитесь к устройству")
            return
        
        try:
            start = self.start_reg.get()
            count = self.reg_count.get()
            
            values = self.instrument.read_registers(start, count, 3, False)
            self.update_register_values(start, values)
            self.status_var.set(f"✅ Таблица обновлена в {time.strftime('%H:%M:%S')}")
            self.log_info(f"🔄 Обновлено {count} регистров")
            
        except Exception as e:
            self.status_var.set(f"❌ Ошибка обновления: {e}")
            self.log_info(f"❌ Ошибка: {e}")
            messagebox.showerror("Ошибка", f"Не удалось обновить данные:\n{e}")
    
    def log_info(self, message):
        """Добавляет сообщение в информационную панель"""
        timestamp = time.strftime("%H:%M:%S")
        self.info_text.insert(tk.END, f"[{timestamp}] {message}\n")
        self.info_text.see(tk.END)
        # Ограничиваем количество строк
        lines = self.info_text.get('1.0', tk.END).count('\n')
        if lines > 100:
            self.info_text.delete('1.0', f'{lines-90}.0')

# ============================================
# ЗАПУСК ПРИЛОЖЕНИЯ
# ============================================

if __name__ == "__main__":
    root = tk.Tk()
    app = ModbusMonitorApp(root)
    root.mainloop()

Exception in thread Thread-7 (monitor_loop):
Traceback (most recent call last):
  File "C:\Users\Iziaslaw\AppData\Local\Temp\ipykernel_6984\1232731151.py", line 254, in monitor_loop
    values = self.instrument.read_registers(start, count, 3, False)
TypeError: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Iziaslaw\AppData\Local\Temp\ipykernel_6984\1232731151.py", line 265, in monitor_loop
    self.root.after(0, lambda: self.status_var.set(f"⚠️ Ошибка чтения: {e}"))
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Python313\Lib\tkinter\__init__.py", line 873, in after
    name = self._register(callit)
  File "D:\Python313\Lib\tkinter\__init__.py", line 1698, in _register
    self.tk.createcommand(name, f)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
RuntimeError: main thread is not in main loop

Duri

In [8]:
import minimalmodbus
import time

class ModbusDevice:
    """Класс для работы с Modbus устройством с обработкой ошибок"""
    
    def __init__(self, port, slave_id=18, baudrate=9600):
        self.port = port
        self.slave_id = slave_id
        self.baudrate = baudrate
        self.instrument = None
        self.available_registers = []
        
    def connect(self):
        """Подключается к устройству"""
        try:
            self.instrument = minimalmodbus.Instrument(self.port, self.slave_id)
            self.instrument.serial.baudrate = self.baudrate
            self.instrument.serial.timeout = 1
            self.instrument.mode = minimalmodbus.MODE_RTU
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения: {e}")
            return False
    
    def read_register_safe(self, register, signed=False):
        """Безопасное чтение регистра с обработкой ошибок"""
        if not self.instrument:
            return None
        
        try:
            value = self.instrument.read_register(register, 0, 3, signed)
            return value
        except minimalmodbus.IllegalRequestError as e:
            print(f"⚠️ Регистр {register}: Неверный адрес (ошибка 02)")
            return None
        except minimalmodbus.NoResponseError:
            print(f"⚠️ Регистр {register}: Нет ответа")
            return None
        except Exception as e:
            print(f"⚠️ Регистр {register}: {e}")
            return None
    
    def discover_registers(self, start=0, end=100):
        """Находит все доступные регистры"""
        print("🔍 Поиск доступных регистров...")
        
        for reg in range(start, end):
            value = self.read_register_safe(reg)
            if value is not None:
                self.available_registers.append((reg, value))
                print(f"  ✅ Регистр {reg:3d}: {value}")
            else:
                # Если несколько регистров подряд недоступны, возможно дальше тоже
                if reg > start + 5:
                    # Проверяем, есть ли еще доступные регистры
                    test = self.read_register_safe(reg + 1)
                    if test is None:
                        break
        
        return self.available_registers
    
    def read_multiple(self, start, count):
        """Читает несколько регистров"""
        if not self.instrument:
            return []
        
        try:
            return self.instrument.read_registers(start, count, 3, False)
        except Exception as e:
            print(f"❌ Ошибка чтения: {e}")
            return []
    
    def close(self):
        """Закрывает соединение"""
        if self.instrument:
            try:
                self.instrument.serial.close()
            except:
                pass

# ============================================
# ИСПОЛЬЗОВАНИЕ
# ============================================

def main():
    # Создаем устройство
    device = ModbusDevice('COM5', 18, 9600)
    
    # Подключаемся
    if not device.connect():
        print("❌ Не удалось подключиться")
        return
    
    print("✅ Подключено успешно!")
    
    # Находим доступные регистры
    registers = device.discover_registers(0, 50)
    
    if registers:
        print(f"\n✅ Найдено {len(registers)} регистров:")
        for reg, value in registers:
            print(f"  Регистр {reg}: {value}")
    
    # Читаем конкретный регистр (например, первый найденный)
    if registers:
        reg_num = registers[0][0]
        value = device.read_register_safe(reg_num)
        print(f"\n📊 Регистр {reg_num}: {value}")
    
    # Закрываем
    device.close()

if __name__ == "__main__":
    main()

✅ Подключено успешно!
🔍 Поиск доступных регистров...
⚠️ Регистр 0: Checksum error in rtu mode: b'1\\' instead of b'14' . The response is: b'\x12\x83\x021\\' (plain response: b'\x12\x83\x021\\')
  ✅ Регистр   1: 4
  ✅ Регистр   2: 1
  ✅ Регистр   3: 0
  ✅ Регистр   4: 0
  ✅ Регистр   5: 0
  ✅ Регистр   6: 0
  ✅ Регистр   7: 1
  ✅ Регистр   8: 0
⚠️ Регистр 9: Checksum error in rtu mode: b'1\\' instead of b'14' . The response is: b'\x12\x83\x021\\' (plain response: b'\x12\x83\x021\\')
⚠️ Регистр 10: Checksum error in rtu mode: b'1\\' instead of b'14' . The response is: b'\x12\x83\x021\\' (plain response: b'\x12\x83\x021\\')

✅ Найдено 8 регистров:
  Регистр 1: 4
  Регистр 2: 1
  Регистр 3: 0
  Регистр 4: 0
  Регистр 5: 0
  Регистр 6: 0
  Регистр 7: 1
  Регистр 8: 0

📊 Регистр 1: 4


In [9]:
import minimalmodbus
import struct
import time
from enum import Enum

class DataType(Enum):
    """Типы данных Modbus"""
    UINT16 = 'uint16'      # 2 байта, без знака
    INT16 = 'int16'        # 2 байта, со знаком
    UINT32 = 'uint32'      # 4 байта, без знака
    INT32 = 'int32'        # 4 байта, со знаком
    FLOAT32 = 'float32'    # 4 байта, IEEE 754
    FLOAT64 = 'float64'    # 8 байт, IEEE 754 (double)
    BIT = 'bit'            # 1 бит
    BITS8 = 'bits8'        # 8 бит (байт)
    BITS16 = 'bits16'      # 16 бит
    ASCII = 'ascii'        # ASCII строка
    BCD = 'bcd'            # Binary Coded Decimal

class ModbusDataTypeConverter:
    """Конвертер типов данных для Modbus"""
    
    @staticmethod
    def get_register_count(data_type):
        """Возвращает количество регистров для типа данных"""
        sizes = {
            DataType.UINT16: 1,
            DataType.INT16: 1,
            DataType.UINT32: 2,
            DataType.INT32: 2,
            DataType.FLOAT32: 2,
            DataType.FLOAT64: 4,
            DataType.BIT: 1,      # но читается как бит
            DataType.BITS8: 1,
            DataType.BITS16: 1,
            DataType.ASCII: 1,     # зависит от длины строки
            DataType.BCD: 1
        }
        return sizes.get(data_type, 1)
    
    @staticmethod
    def decode(registers, data_type, byte_order='big', word_order='big'):
        """
        Декодирует значения из регистров
        
        Args:
            registers: список значений регистров
            data_type: тип данных (DataType)
            byte_order: 'big' или 'little' (порядок байт в слове)
            word_order: 'big' или 'little' (порядок слов для 32/64 бит)
        """
        if not registers:
            return None
        
        # Конвертируем в байты
        if data_type in [DataType.UINT16, DataType.INT16, 
                        DataType.BITS8, DataType.BITS16, DataType.BCD]:
            # 2 байта
            bytes_data = b''
            for reg in registers:
                if byte_order == 'big':
                    bytes_data += struct.pack('>H', reg)
                else:
                    bytes_data += struct.pack('<H', reg)
        
        elif data_type in [DataType.UINT32, DataType.INT32, DataType.FLOAT32]:
            # 4 байта (2 регистра)
            bytes_data = b''
            if word_order == 'big':
                # Сначала старшее слово
                for reg in registers:
                    if byte_order == 'big':
                        bytes_data += struct.pack('>H', reg)
                    else:
                        bytes_data += struct.pack('<H', reg)
            else:
                # Сначала младшее слово
                for reg in reversed(registers):
                    if byte_order == 'big':
                        bytes_data += struct.pack('>H', reg)
                    else:
                        bytes_data += struct.pack('<H', reg)
        
        elif data_type == DataType.FLOAT64:
            # 8 байт (4 регистра)
            bytes_data = b''
            if word_order == 'big':
                for reg in registers:
                    if byte_order == 'big':
                        bytes_data += struct.pack('>H', reg)
                    else:
                        bytes_data += struct.pack('<H', reg)
            else:
                for reg in reversed(registers):
                    if byte_order == 'big':
                        bytes_data += struct.pack('>H', reg)
                    else:
                        bytes_data += struct.pack('<H', reg)
        
        elif data_type == DataType.ASCII:
            # ASCII строка
            bytes_data = b''
            for reg in registers:
                bytes_data += struct.pack('>H', reg)
        
        # Декодируем
        try:
            if data_type == DataType.UINT16:
                return struct.unpack('>H' if byte_order == 'big' else '<H', bytes_data[:2])[0]
            elif data_type == DataType.INT16:
                return struct.unpack('>h' if byte_order == 'big' else '<h', bytes_data[:2])[0]
            elif data_type == DataType.UINT32:
                return struct.unpack('>I' if byte_order == 'big' else '<I', bytes_data[:4])[0]
            elif data_type == DataType.INT32:
                return struct.unpack('>i' if byte_order == 'big' else '<i', bytes_data[:4])[0]
            elif data_type == DataType.FLOAT32:
                return struct.unpack('>f' if byte_order == 'big' else '<f', bytes_data[:4])[0]
            elif data_type == DataType.FLOAT64:
                return struct.unpack('>d' if byte_order == 'big' else '<d', bytes_data[:8])[0]
            elif data_type == DataType.BIT:
                return (registers[0] >> 0) & 0x01
            elif data_type == DataType.BITS8:
                return registers[0] & 0xFF
            elif data_type == DataType.BITS16:
                return registers[0]
            elif data_type == DataType.ASCII:
                # Убираем нулевые байты
                ascii_str = bytes_data.decode('ascii', errors='ignore').strip('\x00')
                return ascii_str
            elif data_type == DataType.BCD:
                # Простое преобразование BCD в десятичное
                return int(str(registers[0]), 16)
        except:
            return None
        
        return None

class FlexibleModbusDevice:
    """Гибкое устройство для работы с разными типами данных"""
    
    def __init__(self, port, slave_id=18, baudrate=9600):
        self.port = port
        self.slave_id = slave_id
        self.baudrate = baudrate
        self.instrument = None
        self.registered_tags = {}  # Словарь с настройками регистров
        
    def connect(self):
        """Подключается к устройству"""
        try:
            self.instrument = minimalmodbus.Instrument(self.port, self.slave_id)
            self.instrument.serial.baudrate = self.baudrate
            self.instrument.serial.timeout = 1
            self.instrument.mode = minimalmodbus.MODE_RTU
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения: {e}")
            return False
    
    def register_tag(self, name, address, data_type, 
                     byte_order='big', word_order='big', 
                     scale=1.0, offset=0.0):
        """
        Регистрирует тег для чтения
        
        Args:
            name: имя тега
            address: начальный адрес регистра
            data_type: тип данных (DataType)
            byte_order: 'big' или 'little'
            word_order: 'big' или 'little'
            scale: множитель
            offset: смещение
        """
        reg_count = ModbusDataTypeConverter.get_register_count(data_type)
        self.registered_tags[name] = {
            'address': address,
            'count': reg_count,
            'type': data_type,
            'byte_order': byte_order,
            'word_order': word_order,
            'scale': scale,
            'offset': offset,
            'value': None,
            'raw_value': None,
            'timestamp': None
        }
        print(f"✅ Зарегистрирован тег: {name} (адрес {address}, тип {data_type.value})")
    
    def read_tag(self, name):
        """Читает значение тега"""
        if name not in self.registered_tags:
            raise ValueError(f"Тег {name} не зарегистрирован")
        
        tag = self.registered_tags[name]
        
        try:
            # Читаем регистры
            registers = self.instrument.read_registers(
                tag['address'], 
                tag['count'], 
                3,  # функция 03
                False  # без знака
            )
            
            # Декодируем
            raw_value = ModbusDataTypeConverter.decode(
                registers,
                tag['type'],
                tag['byte_order'],
                tag['word_order']
            )
            
            # Применяем масштаб и смещение
            if raw_value is not None:
                value = raw_value * tag['scale'] + tag['offset']
                tag['value'] = value
                tag['raw_value'] = raw_value
                tag['timestamp'] = time.time()
                return value
            else:
                tag['value'] = None
                return None
                
        except Exception as e:
            print(f"❌ Ошибка чтения тега {name}: {e}")
            tag['value'] = None
            return None
    
    def read_all_tags(self):
        """Читает все зарегистрированные теги"""
        results = {}
        for name in self.registered_tags:
            results[name] = self.read_tag(name)
        return results
    
    def read_multiple(self, start_address, count, data_type, 
                      byte_order='big', word_order='big'):
        """
        Читает произвольное количество регистров с декодированием
        """
        try:
            registers = self.instrument.read_registers(start_address, count, 3, False)
            return ModbusDataTypeConverter.decode(
                registers,
                data_type,
                byte_order,
                word_order
            )
        except Exception as e:
            print(f"❌ Ошибка чтения: {e}")
            return None
    
    def close(self):
        """Закрывает соединение"""
        if self.instrument:
            try:
                self.instrument.serial.close()
            except:
                pass

# ============================================
# ПРИМЕРЫ ИСПОЛЬЗОВАНИЯ
# ============================================

def example_register_tags():
    """Пример регистрации тегов разных типов"""
    
    device = FlexibleModbusDevice('COM5', 18, 9600)
    
    if not device.connect():
        print("❌ Не удалось подключиться")
        return
    
    print("🔧 Регистрация тегов...")
    
    # Регистрируем теги разных типов
    device.register_tag('temperature', 10, DataType.FLOAT32, 
                       byte_order='big', word_order='big', scale=1.0)
    
    device.register_tag('pressure', 12, DataType.FLOAT32,
                       byte_order='big', word_order='big', scale=0.1)
    
    device.register_tag('status', 14, DataType.UINT16,
                       byte_order='big', word_order='big')
    
    device.register_tag('counter', 15, DataType.UINT32,
                       byte_order='big', word_order='big')
    
    device.register_tag('flags', 17, DataType.BITS16,
                       byte_order='big', word_order='big')
    
    device.register_tag('serial', 18, DataType.ASCII,
                       byte_order='big', word_order='big', count=4)  # 4 регистра для строки
    
    # Читаем все теги
    print("\n📊 Чтение тегов:")
    print("="*50)
    
    for name in device.registered_tags:
        value = device.read_tag(name)
        tag = device.registered_tags[name]
        print(f"  {name:12s}: {value} (raw: {tag['raw_value']})")
        time.sleep(0.1)
    
    device.close()

def example_scan_registers():
    """Сканирование регистров с определением типов"""
    
    device = FlexibleModbusDevice('COM5', 18, 9600)
    
    if not device.connect():
        return
    
    print("🔍 Сканирование регистров...")
    print("="*50)
    
    # Сканируем регистры с разными типами
    for reg in range(0, 50):
        # Пробуем как UINT16
        try:
            value = device.read_multiple(reg, 1, DataType.UINT16)
            if value is not None:
                print(f"  Регистр {reg:3d} (UINT16): {value}")
                
                # Пробуем как INT16
                value_int = device.read_multiple(reg, 1, DataType.INT16)
                if value_int is not None and value_int != value:
                    print(f"           (INT16): {value_int}")
                
                # Пробуем как FLOAT32 (нужно 2 регистра)
                if reg % 2 == 0:
                    value_float = device.read_multiple(reg, 2, DataType.FLOAT32)
                    if value_float is not None:
                        print(f"           (FLOAT32): {value_float:.4f}")
                
                continue
        except:
            pass
        
        # Пробуем как UINT32 (2 регистра)
        try:
            value = device.read_multiple(reg, 2, DataType.UINT32)
            if value is not None:
                print(f"  Регистр {reg:3d} (UINT32): {value}")
                # Пробуем как FLOAT32
                value_float = device.read_multiple(reg, 2, DataType.FLOAT32)
                if value_float is not None:
                    print(f"           (FLOAT32): {value_float:.4f}")
                continue
        except:
            pass
        
        time.sleep(0.05)
    
    device.close()

def example_read_different_types():
    """Чтение одного регистра в разных форматах"""
    
    device = FlexibleModbusDevice('COM5', 18, 9600)
    
    if not device.connect():
        return
    
    reg = 10  # Регистр для теста
    
    print(f"📊 Регистр {reg} в разных форматах:")
    print("="*50)
    
    formats = [
        (DataType.UINT16, 'uint16', 1),
        (DataType.INT16, 'int16', 1),
        (DataType.UINT32, 'uint32', 2),
        (DataType.INT32, 'int32', 2),
        (DataType.FLOAT32, 'float32', 2),
        (DataType.BITS16, 'bits16', 1),
        (DataType.BCD, 'bcd', 1),
    ]
    
    for data_type, name, count in formats:
        try:
            value = device.read_multiple(reg, count, data_type)
            if value is not None:
                print(f"  {name:10s}: {value}")
        except Exception as e:
            print(f"  {name:10s}: Ошибка - {e}")
    
    device.close()

# ============================================
# TINKTER ИНТЕГРАЦИЯ
# ============================================

def create_tkinter_interface():
    """Создает Tkinter интерфейс с выбором типа данных"""
    
    import tkinter as tk
    from tkinter import ttk
    
    root = tk.Tk()
    root.title("Modbus - Гибкий монитор")
    root.geometry("800x600")
    
    # ... (код интерфейса из предыдущего примера, но с добавлением выбора типа данных)
    
    # Пример добавления выбора типа данных
    type_frame = ttk.LabelFrame(root, text="Тип данных", padding=10)
    type_frame.pack(fill=tk.X, padx=10, pady=5)
    
    # Выбор типа данных
    data_type_var = tk.StringVar(value="uint16")
    data_types = [
        ('uint16', 'UINT16 (2 байта)'),
        ('int16', 'INT16 (2 байта)'),
        ('uint32', 'UINT32 (4 байта)'),
        ('int32', 'INT32 (4 байта)'),
        ('float32', 'FLOAT32 (4 байта)'),
        ('float64', 'FLOAT64 (8 байт)'),
        ('ascii', 'ASCII строка'),
        ('bcd', 'BCD')
    ]
    
    for i, (value, label) in enumerate(data_types):
        ttk.Radiobutton(type_frame, text=label, variable=data_type_var, 
                       value=value).grid(row=i//4, column=i%4, padx=5, sticky=tk.W)
    
    # Порядок байт
    ttk.Label(type_frame, text="Порядок байт:").grid(row=3, column=0, padx=5)
    byte_order_var = tk.StringVar(value="big")
    ttk.Combobox(type_frame, textvariable=byte_order_var, 
                values=['big', 'little'], width=10).grid(row=3, column=1, padx=5)
    
    # Масштаб
    ttk.Label(type_frame, text="Масштаб:").grid(row=3, column=2, padx=5)
    scale_var = tk.DoubleVar(value=1.0)
    ttk.Entry(type_frame, textvariable=scale_var, width=10).grid(row=3, column=3, padx=5)
    
    # Смещение
    ttk.Label(type_frame, text="Смещение:").grid(row=3, column=4, padx=5)
    offset_var = tk.DoubleVar(value=0.0)
    ttk.Entry(type_frame, textvariable=offset_var, width=10).grid(row=3, column=5, padx=5)
    
    return root

if __name__ == "__main__":
    # Запускаем примеры
    # example_register_tags()
    # example_scan_registers()
    example_read_different_types()

📊 Регистр 10 в разных форматах:
❌ Ошибка чтения: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения: Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given


In [11]:
import minimalmodbus
import struct
import time
from datetime import datetime
import json
import csv

class DataType:
    """Типы данных Modbus"""
    UINT16 = 'uint16'
    INT16 = 'int16'
    UINT32 = 'uint32'
    INT32 = 'int32'
    FLOAT32 = 'float32'
    FLOAT64 = 'float64'
    BIT = 'bit'
    BITS8 = 'bits8'
    BITS16 = 'bits16'
    ASCII = 'ascii'
    BCD = 'bcd'
    
    @staticmethod
    def get_size(data_type):
        """Возвращает размер в байтах"""
        sizes = {
            'uint16': 2,
            'int16': 2,
            'uint32': 4,
            'int32': 4,
            'float32': 4,
            'float64': 8,
            'bit': 0.125,  # 1 бит
            'bits8': 1,
            'bits16': 2,
            'ascii': 1,    # на символ
            'bcd': 2
        }
        return sizes.get(data_type, 2)
    
    @staticmethod
    def get_registers_count(data_type, count=1):
        """Возвращает количество регистров"""
        if data_type == 'ascii':
            return count  # каждый регистр = 2 символа ASCII
        elif data_type == 'bit':
            return 1
        else:
            return DataType.get_size(data_type) // 2

class ModbusRegisterMap:
    """Класс для работы с картой регистров"""
    
    def __init__(self, port, slave_id=18, baudrate=9600):
        self.port = port
        self.slave_id = slave_id
        self.baudrate = baudrate
        self.instrument = None
        self.registers = {}  # Карта регистров
        self.values = {}     # Текущие значения
        
    def connect(self):
        """Подключается к устройству"""
        try:
            self.instrument = minimalmodbus.Instrument(self.port, self.slave_id)
            self.instrument.serial.baudrate = self.baudrate
            self.instrument.serial.timeout = 1
            self.instrument.mode = minimalmodbus.MODE_RTU
            print(f"✅ Подключено к {self.port}, адрес {self.slave_id}")
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения: {e}")
            return False
    
    def close(self):
        """Закрывает соединение"""
        if self.instrument:
            try:
                self.instrument.serial.close()
                print("✅ Соединение закрыто")
            except:
                pass
    
    def load_register_map(self, map_data):
        """
        Загружает карту регистров
        
        map_data - список словарей с полями:
        {
            'name': 'Температура',
            'address': 10,
            'type': 'float32',
            'scale': 1.0,
            'offset': 0.0,
            'unit': '°C',
            'description': 'Температура датчика'
        }
        """
        self.registers = {}
        for reg in map_data:
            name = reg['name']
            self.registers[name] = {
                'address': reg['address'],
                'type': reg.get('type', 'uint16'),
                'count': reg.get('count', 1),
                'scale': reg.get('scale', 1.0),
                'offset': reg.get('offset', 0.0),
                'unit': reg.get('unit', ''),
                'description': reg.get('description', ''),
                'byte_order': reg.get('byte_order', 'big'),
                'word_order': reg.get('word_order', 'big'),
                'registers_count': DataType.get_registers_count(
                    reg.get('type', 'uint16'),
                    reg.get('count', 1)
                )
            }
            self.values[name] = None
        
        print(f"✅ Загружено {len(self.registers)} регистров")
        return self.registers
    
    def read_register(self, name):
        """Читает значение регистра по имени"""
        if name not in self.registers:
            raise ValueError(f"Регистр '{name}' не найден в карте")
        
        reg = self.registers[name]
        address = reg['address']
        reg_count = reg['registers_count']
        data_type = reg['type']
        byte_order = reg['byte_order']
        word_order = reg['word_order']
        scale = reg['scale']
        offset = reg['offset']
        
        try:
            # Читаем регистры
            registers = self.instrument.read_registers(address, reg_count, 3, False)
            
            # Декодируем в зависимости от типа
            value = self._decode_value(registers, data_type, byte_order, word_order)
            
            if value is not None:
                # Применяем масштаб и смещение
                value = value * scale + offset
                self.values[name] = {
                    'value': value,
                    'raw': registers,
                    'timestamp': datetime.now().isoformat()
                }
                return value
            else:
                self.values[name] = None
                return None
                
        except Exception as e:
            print(f"❌ Ошибка чтения '{name}': {e}")
            self.values[name] = None
            return None
    
    def _decode_value(self, registers, data_type, byte_order, word_order):
        """Декодирует значение из регистров"""
        if not registers:
            return None
        
        # Конвертируем регистры в байты
        bytes_data = b''
        if word_order == 'big':
            for reg in registers:
                if byte_order == 'big':
                    bytes_data += struct.pack('>H', reg)
                else:
                    bytes_data += struct.pack('<H', reg)
        else:
            for reg in reversed(registers):
                if byte_order == 'big':
                    bytes_data += struct.pack('>H', reg)
                else:
                    bytes_data += struct.pack('<H', reg)
        
        try:
            if data_type == 'uint16':
                return struct.unpack('>H' if byte_order == 'big' else '<H', bytes_data[:2])[0]
            elif data_type == 'int16':
                return struct.unpack('>h' if byte_order == 'big' else '<h', bytes_data[:2])[0]
            elif data_type == 'uint32':
                return struct.unpack('>I' if byte_order == 'big' else '<I', bytes_data[:4])[0]
            elif data_type == 'int32':
                return struct.unpack('>i' if byte_order == 'big' else '<i', bytes_data[:4])[0]
            elif data_type == 'float32':
                return struct.unpack('>f' if byte_order == 'big' else '<f', bytes_data[:4])[0]
            elif data_type == 'float64':
                return struct.unpack('>d' if byte_order == 'big' else '<d', bytes_data[:8])[0]
            elif data_type == 'bit':
                return (registers[0] >> 0) & 0x01
            elif data_type == 'bits8':
                return registers[0] & 0xFF
            elif data_type == 'bits16':
                return registers[0]
            elif data_type == 'ascii':
                # Декодируем ASCII строку
                ascii_str = bytes_data.decode('ascii', errors='ignore').strip('\x00')
                return ascii_str
            elif data_type == 'bcd':
                return int(str(registers[0]), 16)
        except:
            return None
        
        return None
    
    def read_all(self):
        """Читает все регистры из карты"""
        results = {}
        for name in self.registers:
            value = self.read_register(name)
            results[name] = value
        return results
    
    def read_group(self, group_name):
        """Читает группу регистров (по префиксу в имени)"""
        results = {}
        for name in self.registers:
            if name.startswith(group_name):
                results[name] = self.read_register(name)
        return results
    
    def get_formatted_value(self, name):
        """Возвращает отформатированное значение с единицами измерения"""
        if name not in self.values or self.values[name] is None:
            return "—"
        
        data = self.values[name]
        value = data['value']
        unit = self.registers[name]['unit']
        
        # Форматируем в зависимости от типа
        reg_type = self.registers[name]['type']
        
        if reg_type in ['float32', 'float64']:
            formatted = f"{value:.3f}"
        elif reg_type in ['uint32', 'int32']:
            formatted = str(int(value))
        elif reg_type == 'ascii':
            formatted = str(value)
        elif reg_type == 'bit':
            formatted = "ON" if value else "OFF"
        elif reg_type == 'bits16':
            formatted = f"0b{int(value):016b}"
        else:
            formatted = str(value)
        
        if unit:
            return f"{formatted} {unit}"
        return formatted
    
    def export_values(self, filename, format='json'):
        """Экспортирует значения в файл"""
        data = {}
        for name, value in self.values.items():
            if value:
                data[name] = {
                    'value': value['value'],
                    'timestamp': value['timestamp'],
                    'unit': self.registers[name]['unit'],
                    'description': self.registers[name]['description']
                }
        
        if format == 'json':
            with open(filename, 'w') as f:
                json.dump(data, f, indent=4)
        elif format == 'csv':
            with open(filename, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Name', 'Value', 'Unit', 'Timestamp', 'Description'])
                for name, val in data.items():
                    writer.writerow([name, val['value'], val['unit'], 
                                   val['timestamp'], val['description']])
        
        print(f"✅ Экспортировано в {filename}")

# ============================================
# ПРИМЕР КАРТЫ РЕГИСТРОВ
# ============================================

def create_register_map():
    """Создает пример карты регистров"""
    
    return [
        # Основные параметры
        {
            'name': 'temperature',
            'address': 301,
            'type': 'float32',
            'scale': 0.1,
            'offset': -40.0,
            'unit': '°C',
            'description': 'Температура датчика'
        },
        {
            'name': 'pressure',
            'address': 302,
            'type': 'float32',
            'scale': 0.01,
            'offset': 0.0,
            'unit': 'MPa',
            'description': 'Давление'
        },
        {
            'name': 'flow_rate',
            'address': 313,
            'type': 'float32',
            'scale': 1.0,
            'offset': 0.0,
            'unit': 'm³/h',
            'description': 'Расход'
        },
        
        # Целочисленные значения
        {
            'name': 'counter_total',
            'address': 201,
            'type': 'uint32',
            'scale': 1.0,
            'offset': 0.0,
            'unit': 'm³',
            'description': 'Счетчик общий'
        },
        {
            'name': 'counter_daily',
            'address': 203,
            'type': 'uint32',
            'scale': 0.001,
            'offset': 0.0,
            'unit': 'm³',
            'description': 'Счетчик суточный'
        },
        {
            'name': 'status_code',
            'address': 205,
            'type': 'uint16',
            'scale': 1.0,
            'offset': 0.0,
            'unit': '',
            'description': 'Код статуса'
        },
        
        # Битовые поля
        {
            'name': 'alarm',
            'address': 2,
            'type': 'bits16',
            'scale': 1.0,
            'offset': 0.0,
            'unit': '',
            'description': 'Биты аварий'
        },
        
        # Строковые данные
        #{
            #'name': 'device_name',
            #'address': 30,
            #'type': 'ascii',
            #'count': 8,  # 8 регистров = 16 символов
            #'scale': 1.0,
            #'offset': 0.0,
            #'unit': '',
            #'description': 'Имя устройства'
        #},
        #{
           #'name': 'firmware_version',
           #'address': 38,
           #'type': 'ascii',
           #'count': 4,  # 4 регистра = 8 символов
           #'scale': 1.0,
           #'offset': 0.0,
           #'unit': '',
           #'description': 'Версия прошивки'
        #}
    ]

# ============================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ============================================

def example_use_register_map():
    """Пример использования карты регистров"""
    
    # Создаем устройство
    device = ModbusRegisterMap('COM5', 18, 9600)
    
    # Подключаемся
    if not device.connect():
        return
    
    # Загружаем карту регистров
    register_map = create_register_map()
    device.load_register_map(register_map)
    
    print("\n📊 Чтение всех регистров:")
    print("="*60)
    
    # Читаем все регистры
    results = device.read_all()
    
    # Выводим с форматированием
    for name, value in results.items():
        formatted = device.get_formatted_value(name)
        description = device.registers[name]['description']
        print(f"  {name:20s}: {formatted:15s} ({description})")
    
    # Экспортируем в файл
    device.export_values('modbus_data.json', 'json')
    device.export_values('modbus_data.csv', 'csv')
    
    # Закрываем
    device.close()

# ============================================
# TINKTER ИНТЕРФЕЙС С КАРТОЙ РЕГИСТРОВ
# ============================================

def create_tkinter_app():
    """Создает Tkinter приложение с картой регистров"""
    
    import tkinter as tk
    from tkinter import ttk, messagebox
    
    class RegisterMonitorApp:
        def __init__(self, root):
            self.root = root
            self.root.title("Modbus Монитор по карте регистров")
            self.root.geometry("1000x700")
            
            self.device = None
            self.register_map = []
            self.update_interval = 1000  # мс
            self.is_running = False
            
            self.create_widgets()
            self.load_default_map()
        
        def create_widgets(self):
            # Панель подключения
            conn_frame = ttk.LabelFrame(self.root, text="Подключение", padding=10)
            conn_frame.pack(fill=tk.X, padx=10, pady=5)
            
            ttk.Label(conn_frame, text="Порт:").grid(row=0, column=0)
            self.port_var = tk.StringVar(value="COM5")
            ttk.Entry(conn_frame, textvariable=self.port_var, width=10).grid(row=0, column=1)
            
            ttk.Label(conn_frame, text="Адрес:").grid(row=0, column=2, padx=(10,0))
            self.slave_var = tk.IntVar(value=18)
            ttk.Entry(conn_frame, textvariable=self.slave_var, width=10).grid(row=0, column=3)
            
            ttk.Label(conn_frame, text="Скорость:").grid(row=0, column=4, padx=(10,0))
            self.baud_var = tk.IntVar(value=9600)
            ttk.Combobox(conn_frame, textvariable=self.baud_var, 
                        values=[2400, 4800, 9600, 19200, 38400, 115200],
                        width=10).grid(row=0, column=5)
            
            self.connect_btn = ttk.Button(conn_frame, text="Подключиться", 
                                         command=self.toggle_connect)
            self.connect_btn.grid(row=0, column=6, padx=10)
            
            # Панель управления
            control_frame = ttk.LabelFrame(self.root, text="Управление", padding=10)
            control_frame.pack(fill=tk.X, padx=10, pady=5)
            
            self.start_btn = ttk.Button(control_frame, text="▶ Старт", 
                                       command=self.start_monitoring, state=tk.DISABLED)
            self.start_btn.pack(side=tk.LEFT, padx=5)
            
            self.stop_btn = ttk.Button(control_frame, text="⏹ Стоп", 
                                      command=self.stop_monitoring, state=tk.DISABLED)
            self.stop_btn.pack(side=tk.LEFT, padx=5)
            
            ttk.Button(control_frame, text="🔄 Обновить", 
                      command=self.read_all).pack(side=tk.LEFT, padx=5)
            
            ttk.Button(control_frame, text="💾 Экспорт CSV", 
                      command=self.export_csv).pack(side=tk.LEFT, padx=5)
            
            ttk.Label(control_frame, text="Интервал (мс):").pack(side=tk.LEFT, padx=(20,5))
            self.interval_var = tk.IntVar(value=1000)
            ttk.Entry(control_frame, textvariable=self.interval_var, width=8).pack(side=tk.LEFT)
            
            # Таблица регистров
            table_frame = ttk.LabelFrame(self.root, text="Регистры", padding=10)
            table_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
            
            self.tree = ttk.Treeview(table_frame, columns=('address', 'type', 'value', 'unit', 'description'),
                                    show='headings', height=20)
            
            self.tree.heading('address', text='Адрес')
            self.tree.heading('type', text='Тип')
            self.tree.heading('value', text='Значение')
            self.tree.heading('unit', text='Ед.изм.')
            self.tree.heading('description', text='Описание')
            
            self.tree.column('address', width=80, anchor=tk.CENTER)
            self.tree.column('type', width=80, anchor=tk.CENTER)
            self.tree.column('value', width=150, anchor=tk.CENTER)
            self.tree.column('unit', width=80, anchor=tk.CENTER)
            self.tree.column('description', width=300)
            
            scrollbar = ttk.Scrollbar(table_frame, orient=tk.VERTICAL, command=self.tree.yview)
            self.tree.configure(yscrollcommand=scrollbar.set)
            
            self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
            scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
            
            # Статус
            self.status_var = tk.StringVar(value="Готов")
            status_bar = ttk.Label(self.root, textvariable=self.status_var, 
                                  relief=tk.SUNKEN, anchor=tk.W)
            status_bar.pack(fill=tk.X, padx=10, pady=5)
        
        def load_default_map(self):
            """Загружает карту регистров"""
            self.register_map = create_register_map()
            self.populate_table()
        
        def populate_table(self):
            """Заполняет таблицу регистрами"""
            for item in self.tree.get_children():
                self.tree.delete(item)
            
            for reg in self.register_map:
                self.tree.insert('', tk.END, 
                               values=(reg['address'], reg['type'], '—', 
                                      reg['unit'], reg['description']),
                               iid=f"reg_{reg['name']}")
        
        def toggle_connect(self):
            if not self.device:
                self.connect_device()
            else:
                self.disconnect_device()
        
        def connect_device(self):
            try:
                self.device = ModbusRegisterMap(
                    self.port_var.get(),
                    self.slave_var.get(),
                    self.baud_var.get()
                )
                
                if self.device.connect():
                    self.device.load_register_map(self.register_map)
                    self.connect_btn.config(text="Отключиться")
                    self.start_btn.config(state=tk.NORMAL)
                    self.status_var.set(f"Подключено к {self.port_var.get()}")
                    messagebox.showinfo("Успех", "Подключение установлено")
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось подключиться: {e}")
                self.device = None
        
        def disconnect_device(self):
            if self.device:
                self.stop_monitoring()
                self.device.close()
                self.device = None
            
            self.connect_btn.config(text="Подключиться")
            self.start_btn.config(state=tk.DISABLED)
            self.status_var.set("Отключено")
        
        def start_monitoring(self):
            if not self.device:
                return
            
            self.is_running = True
            self.start_btn.config(state=tk.DISABLED)
            self.stop_btn.config(state=tk.NORMAL)
            
            self.interval = self.interval_var.get() / 1000.0
            self.monitor_loop()
        
        def stop_monitoring(self):
            self.is_running = False
            self.start_btn.config(state=tk.NORMAL)
            self.stop_btn.config(state=tk.DISABLED)
            self.status_var.set("Мониторинг остановлен")
        
        def monitor_loop(self):
            if not self.is_running:
                return
            
            self.read_all()
            self.root.after(self.interval_var.get(), self.monitor_loop)
        
        def read_all(self):
            if not self.device:
                return
            
            try:
                results = self.device.read_all()
                
                for name, value in results.items():
                    item_id = f"reg_{name}"
                    if self.tree.exists(item_id):
                        formatted = self.device.get_formatted_value(name)
                        current_values = list(self.tree.item(item_id)['values'])
                        current_values[2] = formatted
                        
                        # Подсветка изменений (опционально)
                        self.tree.item(item_id, values=current_values)
                
                self.status_var.set(f"Обновлено в {datetime.now().strftime('%H:%M:%S')}")
                
            except Exception as e:
                self.status_var.set(f"Ошибка: {e}")
        
        def export_csv(self):
            if not self.device:
                messagebox.showwarning("Предупреждение", "Сначала подключитесь")
                return
            
            from tkinter import filedialog
            filename = filedialog.asksaveasfilename(
                defaultextension=".csv",
                filetypes=[("CSV files", "*.csv")]
            )
            
            if filename:
                self.device.export_values(filename, 'csv')
                messagebox.showinfo("Успех", f"Данные сохранены в {filename}")
    
    # Запускаем
    root = tk.Tk()
    app = RegisterMonitorApp(root)
    root.mainloop()

# ============================================
# ЗАПУСК
# ============================================

if __name__ == "__main__":
    # Запуск в консоли
    # example_use_register_map()
    
    # Запуск GUI
    create_tkinter_app()

✅ Подключено к COM5, адрес 18
✅ Загружено 7 регистров
❌ Ошибка чтения 'temperature': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'pressure': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'flow_rate': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'counter_total': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'counter_daily': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'status_code': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'alarm': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'temperature': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'pressure': Instr

In [12]:
import minimalmodbus
import struct
import time
from datetime import datetime
import json
import csv

class DataType:
    """Типы данных Modbus"""
    UINT16 = 'uint16'
    INT16 = 'int16'
    UINT32 = 'uint32'
    INT32 = 'int32'
    FLOAT32 = 'float32'
    FLOAT64 = 'float64'
    BIT = 'bit'
    BITS8 = 'bits8'
    BITS16 = 'bits16'
    ASCII = 'ascii'
    BCD = 'bcd'
    
    @staticmethod
    def get_size(data_type):
        """Возвращает размер в байтах"""
        sizes = {
            'uint16': 2,
            'int16': 2,
            'uint32': 4,
            'int32': 4,
            'float32': 4,
            'float64': 8,
            'bit': 0.125,
            'bits8': 1,
            'bits16': 2,
            'ascii': 1,
            'bcd': 2
        }
        return sizes.get(data_type, 2)
    
    @staticmethod
    def get_registers_count(data_type, count=1):
        """Возвращает количество регистров"""
        if data_type == 'ascii':
            return count
        elif data_type == 'bit':
            return 1
        else:
            return DataType.get_size(data_type) // 2

class ModbusRegisterMap:
    """Класс для работы с картой регистров"""
    
    def __init__(self, port, slave_id=18, baudrate=9600):
        self.port = port
        self.slave_id = slave_id
        self.baudrate = baudrate
        self.instrument = None
        self.registers = {}
        self.values = {}
        
    def connect(self):
        """Подключается к устройству"""
        try:
            self.instrument = minimalmodbus.Instrument(self.port, self.slave_id)
            self.instrument.serial.baudrate = self.baudrate
            self.instrument.serial.timeout = 1
            self.instrument.mode = minimalmodbus.MODE_RTU
            print(f"✅ Подключено к {self.port}, адрес {self.slave_id}")
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения: {e}")
            return False
    
    def close(self):
        """Закрывает соединение"""
        if self.instrument:
            try:
                self.instrument.serial.close()
                print("✅ Соединение закрыто")
            except:
                pass
    
    def load_register_map(self, map_data):
        """
        Загружает карту регистров
        """
        self.registers = {}
        for reg in map_data:
            name = reg['name']
            self.registers[name] = {
                'address': reg['address'],
                'type': reg.get('type', 'uint16'),
                'count': reg.get('count', 1),
                'scale': reg.get('scale', 1.0),
                'offset': reg.get('offset', 0.0),
                'unit': reg.get('unit', ''),
                'description': reg.get('description', ''),
                'byte_order': reg.get('byte_order', 'big'),
                'word_order': reg.get('word_order', 'big'),
                'registers_count': DataType.get_registers_count(
                    reg.get('type', 'uint16'),
                    reg.get('count', 1)
                )
            }
            self.values[name] = None
        
        print(f"✅ Загружено {len(self.registers)} регистров")
        return self.registers
    
    def read_register(self, name):
        """Читает значение регистра по имени"""
        if name not in self.registers:
            raise ValueError(f"Регистр '{name}' не найден в карте")
        
        reg = self.registers[name]
        address = reg['address']
        reg_count = reg['registers_count']
        data_type = reg['type']
        byte_order = reg['byte_order']
        word_order = reg['word_order']
        scale = reg['scale']
        offset = reg['offset']
        
        try:
            # ИСПРАВЛЕНО: правильный вызов read_registers()
            # minimalmodbus.read_registers(registeraddress, numberOfRegisters, functioncode=3, signed=False)
            registers = self.instrument.read_registers(address, reg_count, 3, False)
            
            # Декодируем
            value = self._decode_value(registers, data_type, byte_order, word_order)
            
            if value is not None:
                value = value * scale + offset
                self.values[name] = {
                    'value': value,
                    'raw': registers,
                    'timestamp': datetime.now().isoformat()
                }
                return value
            else:
                self.values[name] = None
                return None
                
        except Exception as e:
            print(f"❌ Ошибка чтения '{name}': {e}")
            self.values[name] = None
            return None
    
    def _decode_value(self, registers, data_type, byte_order, word_order):
        """Декодирует значение из регистров"""
        if not registers:
            return None
        
        # Конвертируем регистры в байты
        bytes_data = b''
        if word_order == 'big':
            for reg in registers:
                if byte_order == 'big':
                    bytes_data += struct.pack('>H', reg)
                else:
                    bytes_data += struct.pack('<H', reg)
        else:
            for reg in reversed(registers):
                if byte_order == 'big':
                    bytes_data += struct.pack('>H', reg)
                else:
                    bytes_data += struct.pack('<H', reg)
        
        try:
            if data_type == 'uint16':
                return struct.unpack('>H' if byte_order == 'big' else '<H', bytes_data[:2])[0]
            elif data_type == 'int16':
                return struct.unpack('>h' if byte_order == 'big' else '<h', bytes_data[:2])[0]
            elif data_type == 'uint32':
                return struct.unpack('>I' if byte_order == 'big' else '<I', bytes_data[:4])[0]
            elif data_type == 'int32':
                return struct.unpack('>i' if byte_order == 'big' else '<i', bytes_data[:4])[0]
            elif data_type == 'float32':
                return struct.unpack('>f' if byte_order == 'big' else '<f', bytes_data[:4])[0]
            elif data_type == 'float64':
                return struct.unpack('>d' if byte_order == 'big' else '<d', bytes_data[:8])[0]
            elif data_type == 'bit':
                return (registers[0] >> 0) & 0x01
            elif data_type == 'bits8':
                return registers[0] & 0xFF
            elif data_type == 'bits16':
                return registers[0]
            elif data_type == 'ascii':
                ascii_str = bytes_data.decode('ascii', errors='ignore').strip('\x00')
                return ascii_str
            elif data_type == 'bcd':
                return int(str(registers[0]), 16)
        except Exception as e:
            print(f"❌ Ошибка декодирования: {e}")
            return None
        
        return None
    
    def read_all(self):
        """Читает все регистры из карты"""
        results = {}
        for name in self.registers:
            value = self.read_register(name)
            results[name] = value
            time.sleep(0.05)  # Небольшая задержка между запросами
        return results
    
    def read_group(self, group_name):
        """Читает группу регистров"""
        results = {}
        for name in self.registers:
            if name.startswith(group_name):
                results[name] = self.read_register(name)
        return results
    
    def get_formatted_value(self, name):
        """Возвращает отформатированное значение"""
        if name not in self.values or self.values[name] is None:
            return "—"
        
        data = self.values[name]
        value = data['value']
        unit = self.registers[name]['unit']
        
        reg_type = self.registers[name]['type']
        
        if reg_type in ['float32', 'float64']:
            formatted = f"{value:.3f}"
        elif reg_type in ['uint32', 'int32']:
            formatted = str(int(value))
        elif reg_type == 'ascii':
            formatted = str(value)
        elif reg_type == 'bit':
            formatted = "ON" if value else "OFF"
        elif reg_type == 'bits16':
            formatted = f"0b{int(value):016b}"
        else:
            formatted = str(value)
        
        if unit:
            return f"{formatted} {unit}"
        return formatted
    
    def export_values(self, filename, format='json'):
        """Экспортирует значения в файл"""
        data = {}
        for name, value in self.values.items():
            if value:
                data[name] = {
                    'value': value['value'],
                    'timestamp': value['timestamp'],
                    'unit': self.registers[name]['unit'],
                    'description': self.registers[name]['description']
                }
        
        if format == 'json':
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
        elif format == 'csv':
            with open(filename, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['Name', 'Value', 'Unit', 'Timestamp', 'Description'])
                for name, val in data.items():
                    writer.writerow([name, val['value'], val['unit'], 
                                   val['timestamp'], val['description']])
        
        print(f"✅ Экспортировано в {filename}")

# ============================================
# ПРИМЕР КАРТЫ РЕГИСТРОВ
# ============================================

def create_register_map():
    """Создает пример карты регистров"""
    
    return [
        # Основные параметры
        {
            'name': 'temperature',
            'address': 301,
            'type': 'float32',
            'scale': 0.1,
            'offset': -40.0,
            'unit': '°C',
            'description': 'Температура датчика'
        },
        {
            'name': 'pressure',
            'address': 302,
            'type': 'float32',
            'scale': 0.01,
            'offset': 0.0,
            'unit': 'MPa',
            'description': 'Давление'
        },
        {
            'name': 'flow_rate',
            'address': 313,
            'type': 'float32',
            'scale': 1.0,
            'offset': 0.0,
            'unit': 'm³/h',
            'description': 'Расход'
        },
        
        # Целочисленные значения
        {
            'name': 'counter_total',
            'address': 201,
            'type': 'uint32',
            'scale': 1.0,
            'offset': 0.0,
            'unit': 'm³',
            'description': 'Счетчик общий'
        },
        {
            'name': 'counter_daily',
            'address': 203,
            'type': 'uint32',
            'scale': 0.001,
            'offset': 0.0,
            'unit': 'm³',
            'description': 'Счетчик суточный'
        },
        {
            'name': 'status_code',
            'address': 205,
            'type': 'uint16',
            'scale': 1.0,
            'offset': 0.0,
            'unit': '',
            'description': 'Код статуса'
        },
        
        # Битовые поля
        {
            'name': 'alarm',
            'address': 2,
            'type': 'bits16',
            'scale': 1.0,
            'offset': 0.0,
            'unit': '',
            'description': 'Биты аварий'
        }
    ]

# ============================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ============================================

def example_use_register_map():
    """Пример использования карты регистров"""
    
    # Создаем устройство
    device = ModbusRegisterMap('COM5', 18, 9600)
    
    # Подключаемся
    if not device.connect():
        return
    
    # Загружаем карту регистров
    register_map = create_register_map()
    device.load_register_map(register_map)
    
    print("\n📊 Чтение всех регистров:")
    print("="*60)
    
    # Читаем все регистры
    results = device.read_all()
    
    # Выводим с форматированием
    for name, value in results.items():
        formatted = device.get_formatted_value(name)
        description = device.registers[name]['description']
        print(f"  {name:20s}: {formatted:15s} ({description})")
    
    # Экспортируем
    # device.export_values('modbus_data.json', 'json')
    # device.export_values('modbus_data.csv', 'csv')
    
    # Закрываем
    device.close()

# ============================================
# TINKTER ИНТЕРФЕЙС
# ============================================

def create_tkinter_app():
    """Создает Tkinter приложение"""
    
    import tkinter as tk
    from tkinter import ttk, messagebox
    
    class RegisterMonitorApp:
        def __init__(self, root):
            self.root = root
            self.root.title("Modbus Монитор по карте регистров")
            self.root.geometry("1000x700")
            
            self.device = None
            self.register_map = []
            self.update_interval = 1000
            self.is_running = False
            
            self.create_widgets()
            self.load_default_map()
        
        def create_widgets(self):
            # Панель подключения
            conn_frame = ttk.LabelFrame(self.root, text="Подключение", padding=10)
            conn_frame.pack(fill=tk.X, padx=10, pady=5)
            
            ttk.Label(conn_frame, text="Порт:").grid(row=0, column=0)
            self.port_var = tk.StringVar(value="COM5")
            ttk.Entry(conn_frame, textvariable=self.port_var, width=10).grid(row=0, column=1)
            
            ttk.Label(conn_frame, text="Адрес:").grid(row=0, column=2, padx=(10,0))
            self.slave_var = tk.IntVar(value=18)
            ttk.Entry(conn_frame, textvariable=self.slave_var, width=10).grid(row=0, column=3)
            
            ttk.Label(conn_frame, text="Скорость:").grid(row=0, column=4, padx=(10,0))
            self.baud_var = tk.IntVar(value=9600)
            ttk.Combobox(conn_frame, textvariable=self.baud_var, 
                        values=[2400, 4800, 9600, 19200, 38400, 115200],
                        width=10).grid(row=0, column=5)
            
            self.connect_btn = ttk.Button(conn_frame, text="Подключиться", 
                                         command=self.toggle_connect)
            self.connect_btn.grid(row=0, column=6, padx=10)
            
            # Панель управления
            control_frame = ttk.LabelFrame(self.root, text="Управление", padding=10)
            control_frame.pack(fill=tk.X, padx=10, pady=5)
            
            self.start_btn = ttk.Button(control_frame, text="▶ Старт", 
                                       command=self.start_monitoring, state=tk.DISABLED)
            self.start_btn.pack(side=tk.LEFT, padx=5)
            
            self.stop_btn = ttk.Button(control_frame, text="⏹ Стоп", 
                                      command=self.stop_monitoring, state=tk.DISABLED)
            self.stop_btn.pack(side=tk.LEFT, padx=5)
            
            ttk.Button(control_frame, text="🔄 Обновить", 
                      command=self.read_all).pack(side=tk.LEFT, padx=5)
            
            ttk.Button(control_frame, text="💾 Экспорт CSV", 
                      command=self.export_csv).pack(side=tk.LEFT, padx=5)
            
            ttk.Label(control_frame, text="Интервал (мс):").pack(side=tk.LEFT, padx=(20,5))
            self.interval_var = tk.IntVar(value=1000)
            ttk.Entry(control_frame, textvariable=self.interval_var, width=8).pack(side=tk.LEFT)
            
            # Таблица регистров
            table_frame = ttk.LabelFrame(self.root, text="Регистры", padding=10)
            table_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
            
            self.tree = ttk.Treeview(table_frame, columns=('address', 'type', 'value', 'unit', 'description'),
                                    show='headings', height=20)
            
            self.tree.heading('address', text='Адрес')
            self.tree.heading('type', text='Тип')
            self.tree.heading('value', text='Значение')
            self.tree.heading('unit', text='Ед.изм.')
            self.tree.heading('description', text='Описание')
            
            self.tree.column('address', width=80, anchor=tk.CENTER)
            self.tree.column('type', width=80, anchor=tk.CENTER)
            self.tree.column('value', width=150, anchor=tk.CENTER)
            self.tree.column('unit', width=80, anchor=tk.CENTER)
            self.tree.column('description', width=300)
            
            scrollbar = ttk.Scrollbar(table_frame, orient=tk.VERTICAL, command=self.tree.yview)
            self.tree.configure(yscrollcommand=scrollbar.set)
            
            self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
            scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
            
            # Статус
            self.status_var = tk.StringVar(value="Готов")
            status_bar = ttk.Label(self.root, textvariable=self.status_var, 
                                  relief=tk.SUNKEN, anchor=tk.W)
            status_bar.pack(fill=tk.X, padx=10, pady=5)
        
        def load_default_map(self):
            self.register_map = create_register_map()
            self.populate_table()
        
        def populate_table(self):
            for item in self.tree.get_children():
                self.tree.delete(item)
            
            for reg in self.register_map:
                self.tree.insert('', tk.END, 
                               values=(reg['address'], reg['type'], '—', 
                                      reg['unit'], reg['description']),
                               iid=f"reg_{reg['name']}")
        
        def toggle_connect(self):
            if not self.device:
                self.connect_device()
            else:
                self.disconnect_device()
        
        def connect_device(self):
            try:
                self.device = ModbusRegisterMap(
                    self.port_var.get(),
                    self.slave_var.get(),
                    self.baud_var.get()
                )
                
                if self.device.connect():
                    self.device.load_register_map(self.register_map)
                    self.connect_btn.config(text="Отключиться")
                    self.start_btn.config(state=tk.NORMAL)
                    self.status_var.set(f"Подключено к {self.port_var.get()}")
                    messagebox.showinfo("Успех", "Подключение установлено")
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось подключиться: {e}")
                self.device = None
        
        def disconnect_device(self):
            if self.device:
                self.stop_monitoring()
                self.device.close()
                self.device = None
            
            self.connect_btn.config(text="Подключиться")
            self.start_btn.config(state=tk.DISABLED)
            self.status_var.set("Отключено")
        
        def start_monitoring(self):
            if not self.device:
                return
            
            self.is_running = True
            self.start_btn.config(state=tk.DISABLED)
            self.stop_btn.config(state=tk.NORMAL)
            
            self.monitor_loop()
        
        def stop_monitoring(self):
            self.is_running = False
            self.start_btn.config(state=tk.NORMAL)
            self.stop_btn.config(state=tk.DISABLED)
            self.status_var.set("Мониторинг остановлен")
        
        def monitor_loop(self):
            if not self.is_running:
                return
            
            self.read_all()
            self.root.after(self.interval_var.get(), self.monitor_loop)
        
        def read_all(self):
            if not self.device:
                return
            
            try:
                results = self.device.read_all()
                
                for name, value in results.items():
                    item_id = f"reg_{name}"
                    if self.tree.exists(item_id):
                        formatted = self.device.get_formatted_value(name)
                        current_values = list(self.tree.item(item_id)['values'])
                        current_values[2] = formatted
                        self.tree.item(item_id, values=current_values)
                
                self.status_var.set(f"Обновлено в {datetime.now().strftime('%H:%M:%S')}")
                
            except Exception as e:
                self.status_var.set(f"Ошибка: {e}")
        
        def export_csv(self):
            if not self.device:
                messagebox.showwarning("Предупреждение", "Сначала подключитесь")
                return
            
            from tkinter import filedialog
            filename = filedialog.asksaveasfilename(
                defaultextension=".csv",
                filetypes=[("CSV files", "*.csv")]
            )
            
            if filename:
                self.device.export_values(filename, 'csv')
                messagebox.showinfo("Успех", f"Данные сохранены в {filename}")
    
    root = tk.Tk()
    app = RegisterMonitorApp(root)
    root.mainloop()

# ============================================
# ЗАПУСК
# ============================================

if __name__ == "__main__":
    # Консольный режим
    # example_use_register_map()
    
    # GUI режим
    create_tkinter_app()

✅ Подключено к COM5, адрес 18
✅ Загружено 7 регистров
❌ Ошибка чтения 'temperature': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'pressure': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'flow_rate': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'counter_total': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'counter_daily': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'status_code': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'alarm': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'temperature': Instrument.read_registers() takes from 3 to 4 positional arguments but 5 were given
❌ Ошибка чтения 'pressure': Instr

In [1]:
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import serial
import serial.tools.list_ports
import threading
import time
from datetime import datetime

class RawBytesTerminal:
    """Простой терминал для отправки и приема сырых байт"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("RS485 Raw Bytes Terminal")
        self.root.geometry("950x750")
        
        # Переменные
        self.serial_port = None
        self.is_connected = False
        self.is_receiving = False
        self.receive_thread = None
        
        # Создаем интерфейс
        self.create_widgets()
        
        # Обновляем список портов
        self.refresh_ports()
        
    def create_widgets(self):
        """Создает все виджеты"""
        
        # ===== Верхняя панель настроек =====
        settings_frame = ttk.LabelFrame(self.root, text="Настройки порта", padding=10)
        settings_frame.pack(fill=tk.X, padx=10, pady=5)
        
        # Порт
        ttk.Label(settings_frame, text="Порт:").grid(row=0, column=0, padx=5)
        self.port_var = tk.StringVar()
        self.port_combo = ttk.Combobox(settings_frame, textvariable=self.port_var, width=15)
        self.port_combo.grid(row=0, column=1, padx=5)
        
        # Скорость
        ttk.Label(settings_frame, text="Скорость:").grid(row=0, column=2, padx=5)
        self.baud_var = tk.StringVar(value="9600")
        baud_combo = ttk.Combobox(settings_frame, textvariable=self.baud_var, 
                                  values=["2400", "4800", "9600", "19200", "38400", "57600", "115200"],
                                  width=10)
        baud_combo.grid(row=0, column=3, padx=5)
        
        # Четность
        ttk.Label(settings_frame, text="Четность:").grid(row=0, column=4, padx=5)
        self.parity_var = tk.StringVar(value="NONE")
        parity_combo = ttk.Combobox(settings_frame, textvariable=self.parity_var,
                                   values=["NONE", "EVEN", "ODD"],
                                   width=8)
        parity_combo.grid(row=0, column=5, padx=5)
        
        # Стоп-биты
        ttk.Label(settings_frame, text="Стоп-биты:").grid(row=0, column=6, padx=5)
        self.stopbits_var = tk.StringVar(value="1")
        stopbits_combo = ttk.Combobox(settings_frame, textvariable=self.stopbits_var,
                                     values=["1", "2"],
                                     width=5)
        stopbits_combo.grid(row=0, column=7, padx=5)
        
        # Таймаут
        ttk.Label(settings_frame, text="Таймаут:").grid(row=0, column=8, padx=5)
        self.timeout_var = tk.StringVar(value="0.5")
        ttk.Entry(settings_frame, textvariable=self.timeout_var, width=8).grid(row=0, column=9, padx=5)
        
        # Кнопка обновления портов
        ttk.Button(settings_frame, text="🔄", width=3, 
                  command=self.refresh_ports).grid(row=0, column=10, padx=5)
        
        # Кнопка подключения
        self.connect_btn = ttk.Button(settings_frame, text="Подключиться", 
                                      command=self.toggle_connection)
        self.connect_btn.grid(row=0, column=11, padx=10)
        
        # ===== Панель отправки =====
        send_frame = ttk.LabelFrame(self.root, text="Отправка", padding=10)
        send_frame.pack(fill=tk.X, padx=10, pady=5)
        
        # Поле ввода HEX
        ttk.Label(send_frame, text="HEX:").grid(row=0, column=0, padx=5, sticky=tk.W)
        self.hex_entry = ttk.Entry(send_frame, width=50, font=('Courier New', 10))
        self.hex_entry.grid(row=0, column=1, padx=5, sticky=tk.W+tk.E)
        self.hex_entry.bind('<Return>', lambda e: self.send_hex())
        
        # Поле ввода ASCII
        ttk.Label(send_frame, text="ASCII:").grid(row=1, column=0, padx=5, sticky=tk.W)
        self.ascii_entry = ttk.Entry(send_frame, width=50, font=('Courier New', 10))
        self.ascii_entry.grid(row=1, column=1, padx=5, sticky=tk.W+tk.E)
        self.ascii_entry.bind('<Return>', lambda e: self.send_ascii())
        
        # Кнопки отправки
        btn_frame = ttk.Frame(send_frame)
        btn_frame.grid(row=0, column=2, rowspan=2, padx=10, sticky=tk.N)
        
        ttk.Button(btn_frame, text="Отправить HEX", 
                  command=self.send_hex).pack(pady=2)
        ttk.Button(btn_frame, text="Отправить ASCII", 
                  command=self.send_ascii).pack(pady=2)
        ttk.Button(btn_frame, text="Отправить файл", 
                  command=self.send_file).pack(pady=2)
        
        # Примеры команд
        ttk.Label(send_frame, text="Примеры:").grid(row=2, column=0, padx=5, sticky=tk.W)
        examples_frame = ttk.Frame(send_frame)
        examples_frame.grid(row=2, column=1, padx=5, sticky=tk.W)
        
        ttk.Button(examples_frame, text="Modbus (адрес 18, рег.0)", 
                  command=lambda: self.hex_entry.insert(0, "12 03 00 00 00 01 4F E5")).pack(side=tk.LEFT, padx=2)
        ttk.Button(examples_frame, text="Тест 55 AA", 
                  command=lambda: self.hex_entry.insert(0, "55 AA")).pack(side=tk.LEFT, padx=2)
        ttk.Button(examples_frame, text="FF FF", 
                  command=lambda: self.hex_entry.insert(0, "FF FF")).pack(side=tk.LEFT, padx=2)
        
        # Настройка весов
        send_frame.columnconfigure(1, weight=1)
        
        # ===== Панель приема =====
        receive_frame = ttk.LabelFrame(self.root, text="Прием", padding=10)
        receive_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        
        # Панель управления приемом
        control_frame = ttk.Frame(receive_frame)
        control_frame.pack(fill=tk.X, pady=(0, 5))
        
        self.receive_btn = ttk.Button(control_frame, text="▶ Начать прием", 
                                      command=self.toggle_receive)
        self.receive_btn.pack(side=tk.LEFT, padx=5)
        
        ttk.Button(control_frame, text="Очистить", 
                  command=self.clear_output).pack(side=tk.LEFT, padx=5)
        
        ttk.Button(control_frame, text="Сохранить лог", 
                  command=self.save_log).pack(side=tk.LEFT, padx=5)
        
        # Кнопка диагностики
        ttk.Button(control_frame, text="🔍 Диагностика", 
                  command=self.run_diagnostic).pack(side=tk.LEFT, padx=5)
        
        # Опции отображения
        self.show_hex_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(control_frame, text="HEX", 
                       variable=self.show_hex_var).pack(side=tk.LEFT, padx=10)
        
        self.show_ascii_var = tk.BooleanVar(value=False)
        ttk.Checkbutton(control_frame, text="ASCII", 
                       variable=self.show_ascii_var).pack(side=tk.LEFT, padx=5)
        
        self.show_bin_var = tk.BooleanVar(value=False)
        ttk.Checkbutton(control_frame, text="BIN", 
                       variable=self.show_bin_var).pack(side=tk.LEFT, padx=5)
        
        self.show_dec_var = tk.BooleanVar(value=False)
        ttk.Checkbutton(control_frame, text="DEC", 
                       variable=self.show_dec_var).pack(side=tk.LEFT, padx=5)
        
        self.show_timestamp_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(control_frame, text="Время", 
                       variable=self.show_timestamp_var).pack(side=tk.LEFT, padx=5)
        
        # Поле вывода
        self.output_text = scrolledtext.ScrolledText(receive_frame, height=20, 
                                                     font=('Courier New', 10))
        self.output_text.pack(fill=tk.BOTH, expand=True)
        
        # Настройка цветов
        self.output_text.tag_config('sent', foreground='blue')
        self.output_text.tag_config('received', foreground='darkgreen')
        self.output_text.tag_config('error', foreground='red')
        self.output_text.tag_config('info', foreground='gray')
        self.output_text.tag_config('warning', foreground='orange')
        self.output_text.tag_config('debug', foreground='purple')
        
        # ===== Статусная строка =====
        self.status_var = tk.StringVar(value="Готов к работе")
        status_bar = ttk.Label(self.root, textvariable=self.status_var, 
                              relief=tk.SUNKEN, anchor=tk.W)
        status_bar.pack(fill=tk.X, padx=10, pady=5)
    
    def refresh_ports(self):
        """Обновляет список доступных портов"""
        ports = serial.tools.list_ports.comports()
        port_list = [port.device for port in ports]
        self.port_combo['values'] = port_list
        if port_list:
            self.port_var.set(port_list[0])
        self.log("Порты обновлены", 'info')
    
    def toggle_connection(self):
        """Подключает/отключает порт"""
        if not self.is_connected:
            self.connect_port()
        else:
            self.disconnect_port()
    
    def connect_port(self):
        """Подключается к порту"""
        try:
            parity_map = {
                'NONE': serial.PARITY_NONE,
                'EVEN': serial.PARITY_EVEN,
                'ODD': serial.PARITY_ODD
            }
            
            self.serial_port = serial.Serial(
                port=self.port_var.get(),
                baudrate=int(self.baud_var.get()),
                parity=parity_map[self.parity_var.get()],
                stopbits=int(self.stopbits_var.get()),
                bytesize=serial.EIGHTBITS,
                timeout=float(self.timeout_var.get()),
                write_timeout=1
            )
            
            self.is_connected = True
            self.connect_btn.config(text="Отключиться")
            self.status_var.set(f"Подключено к {self.port_var.get()}")
            self.log(f"✅ Подключено к {self.port_var.get()}", 'info')
            
            # Автоматически запускаем прием
            if not self.is_receiving:
                self.start_receive()
            
        except Exception as e:
            messagebox.showerror("Ошибка", f"Не удалось подключиться:\n{e}")
            self.log(f"❌ Ошибка подключения: {e}", 'error')
    
    def disconnect_port(self):
        """Отключается от порта"""
        self.is_receiving = False
        if self.receive_thread and self.receive_thread.is_alive():
            self.receive_thread.join(timeout=1)
        
        if self.serial_port and self.serial_port.is_open:
            self.serial_port.close()
        
        self.is_connected = False
        self.connect_btn.config(text="Подключиться")
        self.receive_btn.config(text="▶ Начать прием")
        self.status_var.set("Отключено")
        self.log("🔌 Отключено", 'info')
    
    def send_hex(self):
        """Отправляет данные в HEX формате"""
        if not self.is_connected:
            messagebox.showwarning("Предупреждение", "Сначала подключитесь к порту")
            return
        
        hex_str = self.hex_entry.get().strip()
        if not hex_str:
            return
        
        try:
            hex_str = hex_str.replace(' ', '').replace('\t', '')
            if len(hex_str) % 2 != 0:
                raise ValueError("Нечетное количество символов")
            
            data = bytes.fromhex(hex_str)
            self._send_data(data)
            self.hex_entry.delete(0, tk.END)
            
        except ValueError as e:
            messagebox.showerror("Ошибка", f"Неверный HEX формат:\n{e}")
            self.log(f"❌ Ошибка HEX: {e}", 'error')
    
    def send_ascii(self):
        """Отправляет данные в ASCII формате"""
        if not self.is_connected:
            messagebox.showwarning("Предупреждение", "Сначала подключитесь к порту")
            return
        
        ascii_str = self.ascii_entry.get()
        if not ascii_str:
            return
        
        try:
            data = ascii_str.encode('ascii', errors='ignore')
            self._send_data(data)
            self.ascii_entry.delete(0, tk.END)
            
        except Exception as e:
            self.log(f"❌ Ошибка отправки ASCII: {e}", 'error')
    
    def _send_data(self, data):
        """Отправляет данные в порт"""
        try:
            # Очищаем буфер перед отправкой
            self.serial_port.reset_input_buffer()
            
            bytes_sent = self.serial_port.write(data)
            self.display_sent_data(data, bytes_sent)
            
            self.status_var.set(f"Отправлено {bytes_sent} байт")
            
            # Добавляем задержку для получения ответа
            time.sleep(0.1)
            
        except Exception as e:
            self.log(f"❌ Ошибка отправки: {e}", 'error')
    
    def display_sent_data(self, data, bytes_sent):
        """Отображает отправленные данные"""
        timestamp = datetime.now().strftime("%H:%M:%S") if self.show_timestamp_var.get() else ""
        prefix = f"[{timestamp}] " if timestamp else ""
        
        hex_str = ' '.join(f'{b:02X}' for b in data)
        output = f"{prefix}📤 SENT ({bytes_sent} байт): {hex_str}"
        
        if self.show_ascii_var.get():
            ascii_str = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in data)
            output += f" | ASCII: '{ascii_str}'"
        
        if self.show_bin_var.get():
            bin_str = ' '.join(f'{b:08b}' for b in data)
            output += f" | BIN: {bin_str}"
        
        if self.show_dec_var.get():
            dec_str = ' '.join(str(b) for b in data)
            output += f" | DEC: {dec_str}"
        
        self.output_text.insert(tk.END, output + '\n', 'sent')
        self.output_text.see(tk.END)
    
    def toggle_receive(self):
        """Запускает/останавливает прием"""
        if not self.is_connected:
            messagebox.showwarning("Предупреждение", "Сначала подключитесь к порту")
            return
        
        if not self.is_receiving:
            self.start_receive()
        else:
            self.stop_receive()
    
    def start_receive(self):
        """Запускает прием данных"""
        self.is_receiving = True
        self.receive_btn.config(text="⏹ Остановить прием")
        self.status_var.set("Прием данных...")
        self.log("▶ Начат прием данных", 'info')
        
        self.receive_thread = threading.Thread(target=self.receive_loop, daemon=True)
        self.receive_thread.start()
    
    def stop_receive(self):
        """Останавливает прием"""
        self.is_receiving = False
        self.receive_btn.config(text="▶ Начать прием")
        self.status_var.set("Прием остановлен")
        self.log("⏹ Прием остановлен", 'info')
    
    def receive_loop(self):
        """Цикл приема данных"""
        last_activity = time.time()
        
        while self.is_receiving:
            try:
                if self.serial_port and self.serial_port.is_open:
                    # Читаем все доступные данные
                    if self.serial_port.in_waiting > 0:
                        data = self.serial_port.read(self.serial_port.in_waiting)
                        if data:
                            last_activity = time.time()
                            self.root.after(0, lambda: self.display_received_data(data))
                    else:
                        # Если нет данных и прошло больше 5 секунд, показываем индикатор ожидания
                        if time.time() - last_activity > 5:
                            # Не спамим сообщениями, просто обновляем статус
                            pass
                    
                    time.sleep(0.01)
                    
            except Exception as e:
                self.root.after(0, lambda: self.log(f"❌ Ошибка приема: {e}", 'error'))
                break
    
    def display_received_data(self, data):
        """Отображает полученные данные"""
        if not data:
            return
        
        timestamp = datetime.now().strftime("%H:%M:%S") if self.show_timestamp_var.get() else ""
        prefix = f"[{timestamp}] " if timestamp else ""
        
        hex_str = ' '.join(f'{b:02X}' for b in data)
        output = f"{prefix}📥 RECV ({len(data)} байт): {hex_str}"
        
        if self.show_ascii_var.get():
            ascii_chars = []
            for b in data:
                if 32 <= b <= 126:
                    ascii_chars.append(chr(b))
                elif b == 10:
                    ascii_chars.append('\\n')
                elif b == 13:
                    ascii_chars.append('\\r')
                elif b == 9:
                    ascii_chars.append('\\t')
                elif b == 0:
                    ascii_chars.append('\\0')
                else:
                    ascii_chars.append('.')
            ascii_str = ''.join(ascii_chars)
            output += f" | ASCII: '{ascii_str}'"
        
        if self.show_bin_var.get():
            bin_str = ' '.join(f'{b:08b}' for b in data)
            output += f" | BIN: {bin_str}"
        
        if self.show_dec_var.get():
            dec_str = ' '.join(str(b) for b in data)
            output += f" | DEC: {dec_str}"
        
        self.output_text.insert(tk.END, output + '\n', 'received')
        self.output_text.see(tk.END)
        
        self.status_var.set(f"Получено {len(data)} байт")
    
    def send_file(self):
        """Отправляет файл"""
        if not self.is_connected:
            messagebox.showwarning("Предупреждение", "Сначала подключитесь к порту")
            return
        
        from tkinter import filedialog
        
        filename = filedialog.askopenfilename(
            title="Выберите файл для отправки",
            filetypes=[("Все файлы", "*.*")]
        )
        
        if not filename:
            return
        
        try:
            with open(filename, 'rb') as f:
                data = f.read()
            
            self._send_data(data)
            self.log(f"📁 Отправлен файл: {filename} ({len(data)} байт)", 'info')
            
        except Exception as e:
            self.log(f"❌ Ошибка отправки файла: {e}", 'error')
    
    def clear_output(self):
        """Очищает поле вывода"""
        self.output_text.delete(1.0, tk.END)
    
    def save_log(self):
        """Сохраняет лог в файл"""
        from tkinter import filedialog
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("Log files", "*.log")]
        )
        
        if filename:
            try:
                content = self.output_text.get(1.0, tk.END)
                with open(filename, 'w', encoding='utf-8') as f:
                    f.write(content)
                messagebox.showinfo("Успех", f"Лог сохранен в {filename}")
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось сохранить: {e}")
    
    def run_diagnostic(self):
        """Запускает диагностику порта"""
        if not self.is_connected:
            messagebox.showwarning("Предупреждение", "Сначала подключитесь к порту")
            return
        
        self.log("\n" + "="*60, 'info')
        self.log("🔍 ДИАГНОСТИКА ПОРТА", 'info')
        self.log("="*60, 'info')
        
        try:
            # 1. Информация о порте
            self.log(f"Порт: {self.serial_port.port}", 'debug')
            self.log(f"Скорость: {self.serial_port.baudrate}", 'debug')
            self.log(f"Четность: {self.serial_port.parity}", 'debug')
            self.log(f"Стоп-биты: {self.serial_port.stopbits}", 'debug')
            self.log(f"Таймаут: {self.serial_port.timeout}", 'debug')
            self.log(f"Размер байта: {self.serial_port.bytesize}", 'debug')
            
            # 2. Состояние линий
            self.log(f"\nСостояние линий:", 'debug')
            self.log(f"  CTS: {self.serial_port.cts}", 'debug')
            self.log(f"  DSR: {self.serial_port.dsr}", 'debug')
            self.log(f"  RI: {self.serial_port.ri}", 'debug')
            self.log(f"  CD: {self.serial_port.cd}", 'debug')
            
            # 3. Буферы
            self.log(f"\nБуферы:", 'debug')
            self.log(f"  In waiting: {self.serial_port.in_waiting} байт", 'debug')
            self.log(f"  Out waiting: {self.serial_port.out_waiting} байт", 'debug')
            
            # 4. Тест эха (если есть)
            self.log(f"\nТест эха (отправка байта 0x55):", 'debug')
            self.serial_port.reset_input_buffer()
            self.serial_port.write(b'\x55')
            time.sleep(0.2)
            
            echo = self.serial_port.read(1)
            if echo:
                self.log(f"  ✅ Получен эхо: {echo.hex().upper()}", 'debug')
            else:
                self.log(f"  ⚠️ Нет эха (нормально для RS-485)", 'warning')
            
            # 5. Проверка подключения
            self.log(f"\nПроверка связи:", 'debug')
            self.serial_port.write(b'\x12\x03\x00\x00\x00\x01\x4F\xE5')
            time.sleep(0.2)
            
            response = self.serial_port.read(20)
            if response:
                self.log(f"  ✅ Получен ответ: {response.hex().upper()}", 'debug')
            else:
                self.log(f"  ⚠️ Нет ответа на Modbus запрос", 'warning')
            
            self.log("\n" + "="*60, 'info')
            self.log("✅ Диагностика завершена", 'info')
            
        except Exception as e:
            self.log(f"❌ Ошибка диагностики: {e}", 'error')
    
    def log(self, message, tag='info'):
        """Добавляет сообщение в лог"""
        timestamp = datetime.now().strftime("%H:%M:%S") if self.show_timestamp_var.get() else ""
        prefix = f"[{timestamp}] " if timestamp else ""
        self.output_text.insert(tk.END, f"{prefix}{message}\n", tag)
        self.output_text.see(tk.END)

# ============================================
# ЗАПУСК ПРИЛОЖЕНИЯ
# ============================================

if __name__ == "__main__":
    root = tk.Tk()
    app = RawBytesTerminal(root)
    root.mainloop()